# 71 · PT pathways on the package pseudospace

**The question.** Does notebook 69's PT pathway analysis, run unchanged, give the same picture on the package
pseudospace? The user froze Caleb's package `pseudospace_reconstruction` (stages 01–04) as the method that builds
the pseudospace for both datasets and asked to "switch all results to use calebs package now". Notebook 70 built it
and exported the PT units (`results/package_pseudospace/{hvm,aki}/pt_units.h5ad`).

**The single change.** The units, their raw counts, their PT labels and their position come from the package export.
Everything else is notebook 69's analysis, unchanged:
- the NB gene model with the mt-free offset and the specimen-shape denominator;
- the positional F-test with cameraPR (**primary**);
- the clustering comparator, run through our pipeline as notebook 69 runs it;
- the swaps and their pass rules, the gene sets and the redundancy clusters, the mt exclusion.

**The frozen protocol.** `docs/results/pt-package-pseudospace.md`, section "Notebook 71 protocol: notebook 69's PT
pathway analysis, unchanged, on the package pseudospace", committed before any fit. By the user's decision these
results become primary; the protocol fixes the analysis and states beforehand what counts as a material change from
notebook 69 (rules M1–M7).

| Dataset | Reference | Case | Coordinate |
|---|---|---|---|
| Human vs mouse PT | 2 control mice | 2 cortex sections of 1 human donor | package pseudospace (stage 04), min–max over the common support |
| AKI vs control mouse PT | 2 control mice | 2 AKI mice | package pseudospace (stage 04), min–max over the common support |

**Sections:**
1. Setup, inputs and guards.
2. The gene model and the specimen-shape denominator.
3. The normalized method (positional F-test, primary).
4. The clustering method.
5. Calls and pass rules; level-dominance of the clustering method.
6. Null drift and direction balance (diagnostics; M7).
7. Overlap between the methods, with the proteasome sensitivity.
8. The clustering lists (pre-specified).
9. Comparison with notebook 69: units, genes, pathways, pass rules.
10. Material-change verdicts (M1–M7).
11. The figure.
12. Secondary sections S1–S10 (the comparison page's analyses).
13. Readings of the protocol.
14. Summary.
15. Run record.

**Assumptions.** Each coordinate is treated as exact. Specimens, not units, are the replicates. The two human
"replicates" are two sections of one donor. With two specimens per group every result is descriptive. Under the
package's labels the most injured AKI units leave the PT cohort (notebook 70's label checkpoint): a limitation of
every AKI result here.

## 1 · Setup

In [ ]:
import argparse
import json
import os
import sys
import time
import warnings
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
package_root = results_root / 'package_pseudospace'
upstream13 = results_root / 'minimal_pt_scfates'
T69 = results_root / 'pt_pathways_without_mito'                 # notebook 69: read only (tables and stage cache)
coordinate_file = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'      # 69's SCF13 (unit matching)
mouse_v5_file = results_root / 'mouse_only_v5' / 'all_mouse_tubules_scanpy_dpt.h5ad'        # 69's AKI DPT (unit matching)
nb12_universe_file = results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv'  # 69's AKI pool (69's sets)
nb68_tables = results_root / 'pt_species_level_control' / 'tables'                           # S3: atlas columns
census_dir = data_root / 'external' / 'census_pt_segments'                                    # S7, S8
lake_cache_dir = results_root / 'pt_pathway_final' / 'scfates' / 'stage_cache'                # S7, S8 (notebook 37)
lake_h5ad = data_root / 'external' / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'             # S7 cache check
output = results_root / 'pt_pathways_package_pseudospace'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
DATASETS = ['human_vs_mouse', 'aki_vs_control']
SHORT_DS = {'human_vs_mouse': 'hvm', 'aki_vs_control': 'aki'}
EXPORT = {ds: package_root / SHORT_DS[ds] / 'pt_units.h5ad' for ds in DATASETS}
FINGERPRINTS = {'human_vs_mouse': 'efafc9ac2e99', 'aki_vs_control': 'bb4a80c1b484'}   # confirmed pass-1 labels (nb70)
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
REQUIRED = [*EXPORT.values(), upstream13 / 'ortholog_map_used.csv', upstream13 / 'cross_species_pt_scfates.h5ad',
            coordinate_file, mouse_v5_file, nb12_universe_file, nb68_tables / 'gene_level.csv',
            T69 / 'tables' / 'method_table.csv', T69 / 'tables' / 'check_d_overlap.csv',
            T69 / 'tables' / 'check_a_null_drift.csv', T69 / 'tables' / 'check_b_direction_balance.csv',
            T69 / 'tables' / 'gene_drift_before_after.csv',
            *[T69 / 'tables' / f'{kind}_{ds}.csv' for kind in ('redundancy_clusters', 'gene_statistics') for ds in DATASETS],
            *[T69 / 'tables' / f'pathway_tests_{ds}_cameraPR_species.csv' for ds in DATASETS],
            census_dir / 'human_pt_segment_pseudobulk_counts.csv.gz', census_dir / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            census_dir / 'census_symbol_audit.csv', lake_h5ad,
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in LIBRARIES]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 13, 37, 68, 69 and 70 first)')

NOTEBOOK_LOGIC_VERSION = '71.package.1'   # Bump when a cached calculation changes.
MT_PREFIX = 'mt-'                                # mitochondrially encoded genes (mouse symbols)
PROTEASOME = ['Psma3', 'Psma4', 'Psmb5', 'Psmb6', 'Psmc2', 'Psmc6', 'Psmd11', 'Psmd14']   # sensitivity of section 7
INJURY = ['Havcr1', 'Lcn2', 'Mki67', 'Top2a']    # G8 (reported only)
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05      # discovery level and notebook 31's relabeling rule
CALIBRATION_MAX = 1.5 * ALPHA                    # notebook 61's criterion 2: share of relabeled p <= 0.05
CAMERA_COR = .01                                 # limma's default inter.gene.cor (notebook 67)
BASIS_DF, GRID_POINTS = 6, 101                   # the shared gene model
DF1 = 6                                          # numerator: group x basis
DF2 = {'condition': 12, 'relabeling': 6}         # denominator: specimen-within-group x basis (asserted)
GENE_FDR = .05                                   # significant genes on the normalized F
NULL_Q = .5                                      # null genes: q > 0.5
DRIFT_SHARE = .2                                 # d_g: mean delta over the last 20% of the grid minus the first 20%
DRIFT_LIMIT = .05                                # M7: |median d_g of 71's own null genes| <= 0.05 (human vs mouse)
SET_SIZES = (10, 300)
REDUNDANCY_JACCARD = .5                          # notebook 66's redundancy rule
ZONE_MARKERS = {'PT-S1': ['Slc5a12'], 'PT-S2': ['Slc13a3'], 'PT-S3': ['Slc16a9']}   # notebook 66
ZONE_SHORT = {'PT-S1': 'S1', 'PT-S2': 'S2', 'PT-S3': 'S3'}
NODE_SPACING = 1 / 29                            # notebook 66, inherited unchanged (protocol section 3)
EXTERNAL_COMMIT = '59da04b'                      # pseudospace_reconstruction, as in notebook 69
L1_LIMIT, OVERLAP_LIMIT, M6_SHARE = .5, .2, .5   # M4, M5 and M6
TOP5_69 = ['MSigDB_Hallmark_2020::Fatty Acid Metabolism',
           'Reactome_2022::Transport Of Inorganic Cations/Anions And Amino Acids/Oligopeptides R-HSA-425393',
           'KEGG_2019_Mouse::Valine, leucine and isoleucine degradation', 'MSigDB_Hallmark_2020::Xenobiotic Metabolism',
           'KEGG_2019_Mouse::Arginine and proline metabolism']                      # M6: notebook 69's five strongest
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))   # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import distribution, version
from IPython.display import display

direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown'), 'url': direct_url.get('url', '')}
assert EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), f'pseudospace_reconstruction is not commit {EXTERNAL_COMMIT}: {EXTERNAL}'

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
N_COLOR, B_COLOR = '#1b8a5a', '#eb6834'          # notebook 67's normalized and notebook 66's clustering colours
BEFORE_COLOR, AFTER_COLOR = MUTED, N_COLOR       # notebook 69: dashed muted ink; notebook 71: solid green
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output, '· parallel workers:', N_JOBS, '· pseudospace_reconstruction', EXTERNAL['version'],
      EXTERNAL['commit'][:7])

### 1.1 · Inputs and guards

**Builders** (`pseudospace/package_inputs.py`): notebook 69's rules applied to the export.
- **Human vs mouse** (`hvm_inputs`, `pt_inputs`' dict): the library is the total over the shared orthologs without
  the mt genes; the universe is measured in both, not mt, equal-specimen detection ≥ 2%; pathways with 10–300 tested
  members, mapped through the ortholog map.
- **AKI** (`aki_inputs`, notebook 69's cell): the ortholog panel measured in the kept units; the library without the
  mt genes; the pool is the human vs mouse universe with the mt genes kept (as 69 passed notebook 12's universe);
  detection ≥ 2%, minus mt; pathways restricted from the human vs mouse ones with the same 10–300 rule.
- Both: each specimen's [1st, 99th] percentile common support; position = the stage-04 value min–max scaled over the
  kept units (protocol section 3); the log-normalized matrix keeps the full library and feeds only the zone cuts.

**Guards (not cached):**
- G1: the export contract (`read_pt_units`); the provenance is notebook 70's, at package commit 59da04b, with the
  confirmed pass-1 fingerprints (human vs mouse `efafc9ac2e99`, AKI `bb4a80c1b484`).
- G2: in every specimen the segment medians increase S1 < S2 < S3 along the analysed position.
- G3: the mt genes are the 13 `mt-` symbols of the ortholog map; how many leave each library and universe; each
  library falls by exactly their counts; nothing else moves.
- G8 (reported only): whether the universes admit Havcr1, Lcn2, Mki67 and Top2a.

In [ ]:
import anndata as ad
from scipy import sparse

from pseudospace.package_inputs import aki_inputs, hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import common_grid, partition_groups
from pseudospace.pathways import build_pathway_membership
from pseudospace.stage_cache import cached_payload, digest, stage_key

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith(MT_PREFIX))
assert len(MT) == 13, MT
UNITS = {ds: read_pt_units(EXPORT[ds], SHORT_DS[ds]) for ds in DATASETS}        # G1: the contract
guard_log, PROVENANCE = [], {}
for ds, units in UNITS.items():
    prov = units.uns['provenance']
    assert 'dev only' not in json.dumps(prov, default=str), 'the ws71 development object is not an export'
    assert str(prov['notebook_logic_version']).startswith('70.'), prov['notebook_logic_version']
    assert str(prov['package']['commit']).startswith(EXTERNAL_COMMIT), prov['package']
    fingerprint = prov['fingerprints']['pass1']['membership_sha1']
    assert fingerprint == FINGERPRINTS[ds], (ds, fingerprint)
    PROVENANCE[ds] = {'notebook': str(prov['notebook']), 'logic': str(prov['notebook_logic_version']),
                      'package_commit': str(prov['package']['commit']), 'pass1_fingerprint': fingerprint,
                      'pass2_fingerprint': str(prov['fingerprints']['pass2']['membership_sha1']),
                      'units': int(units.n_obs), 'genes': int(units.n_vars)}
guard_log.append('G1: export contract; provenance notebook 70 at package 59da04b with the confirmed pass-1 fingerprints')

kept_h = hvm_inputs(UNITS['human_vs_mouse'], data_root, orthologs)                     # mt kept: the AKI pool, G3
hm = hvm_inputs(UNITS['human_vs_mouse'], data_root, orthologs, exclude_genes=MT)
kept_a = aki_inputs(UNITS['aki_vs_control'], data_root, orthologs, kept_h['genes'])
ak = aki_inputs(UNITS['aki_vs_control'], data_root, orthologs, kept_h['genes'], exclude_genes=MT)
BUILT = {'human_vs_mouse': (kept_h, hm), 'aki_vs_control': (kept_a, ak)}
TITLES = {'human_vs_mouse': ('Human vs mouse PT', 'human', 'mouse'), 'aki_vs_control': ('AKI vs control mouse PT', 'AKI', 'control')}
DATA, LIBRARY_ROWS = {}, []
for ds, (kept, new) in BUILT.items():
    # G3: only the mt genes leave the library and the universe; positions, specimens, segments and sets do not move
    assert new['genes'] == [g for g in kept['genes'] if g not in MT]
    assert set(new['excluded_library']) <= set(MT) and set(new['excluded_universe']) <= set(MT)
    names = UNITS[ds].var_names
    mask = UNITS[ds].obs_names.isin(new['obs'].index)
    mt_cols = names.get_indexer(new['excluded_library'])
    mt_counts = np.asarray(UNITS[ds].X[mask][:, mt_cols].sum(axis=1)).ravel()
    np.testing.assert_allclose(new['library'], kept['library'] - mt_counts, rtol=0, atol=1e-6)
    np.testing.assert_allclose(new['library_with_excluded'], kept['library'], rtol=0, atol=1e-6)
    index = pd.Index(kept['genes'])
    assert (new['counts'].tocsr() != kept['counts'].tocsr()[:, index.get_indexer(new['genes'])]).nnz == 0
    for key in ('position', 'specimen', 'segment', 'case'):
        np.testing.assert_array_equal(new[key], kept[key])
    assert new['gene_sets'] == kept['gene_sets'], 'the pathways changed on the reduced universe'
    # G2: segment medians increase along the analysed position in every specimen
    medians = pd.DataFrame({'position': new['position'], 'specimen': new['specimen'], 'segment': new['segment']}) \
        .groupby(['specimen', 'segment']).position.median().unstack()[['PT-S1', 'PT-S2', 'PT-S3']]
    assert (medians['PT-S1'] < medians['PT-S2']).all() and (medians['PT-S2'] < medians['PT-S3']).all(), medians
    title, case_name, reference_name = TITLES[ds]
    DATA[ds] = {
        'title': title, 'case_name': case_name, 'reference_name': reference_name, 'coordinate': 'package pseudospace',
        'counts': new['counts'].tocsr(), 'library': np.asarray(new['library'], float),
        'library_with_mt': np.asarray(new['library_with_excluded'], float), 'position': np.asarray(new['position'], float),
        'position_raw': np.asarray(new['position_raw'], float), 'specimen': new['specimen'].astype(str),
        'segment': pd.Series(new['segment']).str.replace('PT-', '', regex=False).to_numpy(),
        'case': np.asarray(new['case'], bool), 'genes': list(new['genes']), 'sets': new['gene_sets'],
        'lognorm': new['Y'].tocsr(), 'mt_universe': list(new['excluded_universe']), 'mt_library': list(new['excluded_library']),
        'obs': new['obs'], 'covariates': new['covariates'], 'n_units': new['n_units'], 'support_raw': new['support_raw'],
        'kept_genes': list(kept['genes']), 'segment_medians': medians}
    guard_log.append(f"{title}: G2 segment medians S1 < S2 < S3 in every specimen; G3 {len(new['excluded_library'])} mt genes out "
                     f"of the library and {len(new['excluded_universe'])} out of the universe, library fell by exactly their counts")
del kept_h, hm, kept_a, ak, BUILT
assert list(DATA) == DATASETS

PARTITIONS, SHORT = {}, {}


def group_one(ds, label):
    """The specimens in group 1 of a partition (the case specimens for the condition split)."""
    d = DATA[ds]
    return sorted(set(d['specimen'][PARTITIONS[ds][label][0] == 1]))


def slug(label):
    return label.replace(':', '_').replace('+', '_')


for ds, d in DATA.items():
    assert d['counts'].shape == d['lognorm'].shape == (len(d['position']), len(d['genes']))
    assert all(set(v) <= set(d['genes']) for v in d['sets'].values())
    assert (d['library'] > 0).all() and d['position'].min() == 0 and d['position'].max() == 1
    PARTITIONS[ds] = partition_groups(d['specimen'], d['case'])
    labels = list(PARTITIONS[ds])
    assert labels[0] == 'species'
    SHORT[ds] = dict(zip(labels, ['condition', 'relabeling 1', 'relabeling 2']))
    d['input_key'] = {'counts': digest(d['counts']), 'library': d['library'], 'position': d['position'],
                      'specimen': d['specimen'], 'genes': d['genes']}
    d['log_mean_count'] = np.log(np.asarray(d['counts'].mean(axis=0)).ravel())    # notebook 67's squeezeVar covariate
    d['case_specimens'] = sorted(set(d['specimen'][d['case']]))
    d['reference_specimens'] = sorted(set(d['specimen'][~d['case']]))
    LIBRARY_ROWS.append({'dataset': d['title'], 'mt genes out of the library': len(d['mt_library']),
                         'mt genes out of the universe': len(d['mt_universe']),
                         'mt share of the full library (all units)': 1 - d['library'].sum() / d['library_with_mt'].sum(),
                         'genes (mt kept → out)': f"{len(d['kept_genes']):,} → {len(d['genes']):,}", 'pathways': len(d['sets'])})
    # Tables for later notebooks (72 reads the universe and the gene sets)
    d['covariates'].rename_axis('gene').to_csv(tables / f'universe_{ds}.csv')
    (tables / f'gene_sets_{ds}.json').write_text(json.dumps(d['sets'], indent=0))

# G8 (reported only): the injury markers
units_a = UNITS['aki_vs_control']
kept_units = units_a[units_a.obs_names.isin(DATA['aki_vs_control']['obs'].index)]
INJURY_ROWS = []
for gene in INJURY:
    row = {'gene': gene, 'in the package AKI matrix': gene in units_a.var_names,
           'in the 1:1 ortholog map': gene in set(orthologs.mouse_symbol), 'in the human vs mouse universe': gene in DATA['human_vs_mouse']['kept_genes'],
           'in the AKI universe': gene in DATA['aki_vs_control']['genes']}
    if row['in the package AKI matrix']:
        x = np.asarray(kept_units[:, gene].X.todense()).ravel() > 0
        per = pd.Series(x).groupby(kept_units.obs['sample'].astype(str).to_numpy()).mean()
        row.update({f'detection {s}': float(v) for s, v in per.items()})
        row['equal-specimen detection (AKI units)'] = float(per.mean())
        row['admitted by the AKI detection rule alone'] = bool(per.mean() >= .02)
    INJURY_ROWS.append(row)
INJURY_TABLE = pd.DataFrame(INJURY_ROWS).set_index('gene')
INJURY_TABLE.to_csv(tables / 'g8_injury_markers.csv')
del units_a, kept_units

INPUT_TABLE = pd.DataFrame({d['title']: {
    'units in the export': d['n_units']['before support'], 'units in the common support': d['n_units']['after support'],
    'raw pseudospace of the support': f"{d['support_raw'][0]:.4f}–{d['support_raw'][1]:.4f}",
    'genes': len(d['genes']), 'pathways': len(d['sets']),
    'reference specimens': ', '.join(f"{s} ({int((d['specimen'] == s).sum()):,})" for s in d['reference_specimens']),
    'case specimens': ', '.join(f"{s} ({int((d['specimen'] == s).sum()):,})" for s in d['case_specimens']),
    **{f'{SHORT[ds][label]} (one side)': ' + '.join(group_one(ds, label)) for label in list(PARTITIONS[ds])[1:]},
    'mt genes out of the library': ', '.join(d['mt_library']), 'mt genes out of the universe': len(d['mt_universe']),
    'pass-1 fingerprint': PROVENANCE[ds]['pass1_fingerprint']}
    for ds, d in DATA.items()})
INPUT_TABLE.to_csv(tables / 'inputs.csv')
display(INPUT_TABLE)
display(pd.DataFrame(LIBRARY_ROWS).set_index('dataset').round(4))
display(pd.concat({DATA[ds]['title']: DATA[ds]['segment_medians'] for ds in DATASETS}).round(3))
display(INJURY_TABLE)
print('\n'.join('· ' + line for line in guard_log))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

### 1.2 · Redundancy clusters

Notebook 66's rule (the package's `cluster_gene_sets`: 1 − Jaccard of membership within the tested genes, average
linkage, cut at mean Jaccard 0.5, each cluster named by its medoid), on notebook 71's sets. **Guard (G6):** the
package's gene-set builder and ours agree on the tested sets and their members.

In [ ]:
from pseudospace_reconstruction.compare.pathways import cluster_gene_sets
from pseudospace_reconstruction.io.gene_sets import build_gene_sets

EXT_SETS, REDUNDANCY = {}, {}
for ds, d in DATA.items():
    sets = build_gene_sets({lib: {p.split('::', 1)[1]: v for p, v in d['sets'].items() if p.startswith(lib + '::')}
                            for lib in LIBRARIES}, d['genes'], min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
    ids = [f'{lib}::{name}' for lib, name in zip(sets.library, sets.pathway)]
    assert sorted(ids) == sorted(d['sets']), "the package's gene-set builder and ours disagree on the tested pathways"
    assert all(set(sets.member_names(i)) == set(d['sets'][p]) for i, p in enumerate(ids)), 'members differ'
    EXT_SETS[ds] = sets
    clusters = cluster_gene_sets(sets.membership(), sets.names, REDUNDANCY_JACCARD)
    clusters.index = pd.Index(ids, name='pathway_id')
    medoid = clusters[clusters.is_representative].reset_index().set_index('cluster_id').pathway_id
    clusters['representative'] = clusters.cluster_id.map(medoid)
    REDUNDANCY[ds] = clusters.reindex(list(d['sets']))
    REDUNDANCY[ds].to_csv(tables / f'redundancy_clusters_{ds}.csv')
guard_log.append("G6: the package's gene-set builder and ours agree on the tested sets and members, both datasets")
print({DATA[ds]['title']: f"{len(DATA[ds]['sets']):,} pathways in {REDUNDANCY[ds].cluster_id.nunique():,} redundancy clusters"
       for ds in DATASETS})


def representatives(ds, calls):
    """The redundancy clusters a call set touches, named by their medoids."""
    return set(REDUNDANCY[ds].representative.reindex(sorted(calls)))


NAMES = {ds: pd.Series({p: p.split('::', 1)[1] for p in DATA[ds]['sets']}).str.replace(r' R-HSA-\d+$', '', regex=True)
         for ds in DATASETS}

## 2 · The gene model and the specimen-shape denominator

Notebook 69's section 2, unchanged:
- **Gene model** (`standard_gene_model`): an NB GLM on raw counts with offset log(library without mt reads); a 6-df
  cubic B-spline with knots at the position quartiles; prior weights wᵢ = n / (2 · J · n_j); the 101-point grid. α is
  estimated per gene under the condition split's M_full and reused for both relabelings, where the condition's own
  shape enters as a nuisance term.
- **Denominator** (`specimen_shape_lr`): LR_spec = D(M_full) − D(M_spec), with the same α, weights and offset.

**Guards (G4):** df₂ = 12 for the condition split and 6 for each relabeling, from the design ranks; the specimen
curves are identical in all three partitions; the grid is `common_grid`.

In [ ]:
from pseudospace import nb_gam, pathway_calibration, pathway_pipelines, stats_gam
from pseudospace.pathway_pipelines import GeneModel, specimen_shape_lr, standard_gene_model

cache = output / 'stage_cache'
code_key = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in (nb_gam, pathway_calibration, pathway_pipelines, stats_gam)])
SECONDS = {}   # uncached compute time of every cached stage


def cached(stage, compute, *, params=None, inputs=None, key=None):
    """cached_payload under this notebook's root and code; keeps the stage's uncached compute time."""
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=key or code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def model_payload(model):
    out = {f'stat|{c}': model.statistics[c].to_numpy() for c in model.statistics}
    out.update(grid=model.grid, knots=model.knots, reference=model.reference, case=model.case, delta=model.delta,
               se=model.se, group_names=model.groups.index.to_numpy().astype(str), group_values=model.groups.to_numpy(),
               alpha=np.asarray(model.dispersion['alpha'], float), alpha_converged=np.asarray(model.dispersion['converged'], bool))
    for name, curve in model.specimen_curves.items():
        out[f'curve|{name}'] = curve
        out[f'converged|{name}'] = model.specimen_converged[name].to_numpy()
        out[f'separated|{name}'] = model.specimen_separated[name].to_numpy()
    return out


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


MODELS, DENOM = {}, {}
for ds, d in DATA.items():
    MODELS[ds], DENOM[ds] = {}, {}
    for label, (group, nuisance) in PARTITIONS[ds].items():
        dispersion = None if label == 'species' else MODELS[ds]['species'].dispersion
        payload = cached(f'gene_model_{ds}_{slug(label)}', lambda d=d, group=group, nuisance=nuisance, dispersion=dispersion:
                         model_payload(standard_gene_model(d['counts'], d['library'], d['position'], group, d['specimen'],
                                                           genes=d['genes'], nuisance_shape=nuisance, dispersion=dispersion,
                                                           basis_df=BASIS_DF, n_jobs=N_JOBS)),
                         params={'basis_df': BASIS_DF, 'grid_points': GRID_POINTS, 'label': label},
                         inputs={**d['input_key'], 'group': group, 'nuisance': nuisance,
                                 'alpha': None if dispersion is None else dispersion['alpha']})
        MODELS[ds][label] = model_from(payload, d['genes'])
        alpha = MODELS[ds]['species'].dispersion
        DENOM[ds][label] = cached(f'specimen_shape_{ds}_{slug(label)}',
                                  lambda d=d, group=group, nuisance=nuisance, alpha=alpha:
                                  specimen_shape_lr(d['counts'], d['library'], d['position'], group, d['specimen'], alpha,
                                                    nuisance_shape=nuisance, basis_df=BASIS_DF, n_jobs=N_JOBS),
                                  params={'basis_df': BASIS_DF, 'label': label},
                                  inputs={**d['input_key'], 'group': group, 'nuisance': nuisance, 'alpha': alpha['alpha'],
                                          'alpha_converged': alpha['converged']})

BEFORE_MODEL = {ds: stored(T69, f'gene_model_{ds}_species') for ds in DATASETS}   # notebook 69's condition split
GENES69 = {ds: pd.read_csv(T69 / 'tables' / f'gene_statistics_{ds}.csv', header=[0, 1], index_col=0).index.astype(str).tolist()
           for ds in DATASETS}
for ds in DATASETS:
    condition = MODELS[ds]['species']
    assert len(condition.grid) == GRID_POINTS
    np.testing.assert_allclose(condition.grid, common_grid(DATA[ds]['position'], DATA[ds]['specimen'], GRID_POINTS), rtol=0, atol=1e-12)
    for label, model in MODELS[ds].items():
        expected = DF2['condition' if label == 'species' else 'relabeling']
        assert int(DENOM[ds][label]['df2']) == expected, (ds, label, int(DENOM[ds][label]['df2']))
        np.testing.assert_array_equal(model.grid, condition.grid)
        for name, curve in model.specimen_curves.items():
            np.testing.assert_allclose(curve, condition.specimen_curves[name], rtol=1e-9, atol=1e-9)
guard_log.append(f"G4: df2 = {DF2['condition']} (condition split) and {DF2['relabeling']} (each relabeling); specimen curves "
                 "identical in all three partitions; grid = common_grid, both datasets")

MODEL_ROWS = []
for ds in DATASETS:
    both = pd.Index(GENES69[ds]).intersection(DATA[ds]['genes'])
    alpha69 = pd.Series(BEFORE_MODEL[ds]['alpha'], index=GENES69[ds])
    for label, model in MODELS[ds].items():
        den = DENOM[ds][label]
        MODEL_ROWS.append({
            'dataset': DATA[ds]['title'], 'partition': SHORT[ds][label], 'median α': float(np.median(model.statistics.alpha)),
            'median α, notebook 69 (genes in both)': float(alpha69[both].median()) if label == 'species' else np.nan,
            'median α, notebook 71 (genes in both)': float(model.statistics.alpha[both].median()) if label == 'species' else np.nan,
            'non-converged (median-filled)': int((~model.statistics.converged).sum()),
            'separated (flagged)': int(model.statistics.separated.sum()), 'median T_spatial': float(model.statistics.T_spatial.median()),
            'df2': int(den['df2']), 'median LR_spec / df2': float(np.median(np.asarray(den['LR_spec'], float) / int(den['df2']))),
            'grid': f'{model.grid[0]:.3f}–{model.grid[-1]:.3f}',
            'gene model seconds': round(SECONDS[f'gene_model_{ds}_{slug(label)}'], 1),
            'denominator seconds': round(SECONDS[f'specimen_shape_{ds}_{slug(label)}'], 1)})
MODEL_TABLE = pd.DataFrame(MODEL_ROWS)
MODEL_TABLE.to_csv(tables / 'gene_model_summary.csv', index=False)
display(MODEL_TABLE.round(3))
print('\n'.join('· ' + line for line in guard_log))

## 3 · The normalized method (positional F-test; primary)

Notebook 67's primary, unchanged: s² = LR_spec / df₂ moderated by `limma::squeezeVar` with the log mean raw count per
unit as covariate; F = (`T_spatial` / 6) / s²_post with df (6, df₂ + d₀); z = Φ⁻¹(1 − p)
(`pathway_pipelines.moderated_f`). The gene-level q is the BH of p over all tested genes.

Each pathway is tested by `limma::cameraPR` on z (`inter.gene.cor` 0.01, `use.ranks` FALSE). A call is Up with
FDR ≤ 0.05. For calibration the one-sided p is p/2 when Up, else 1 − p/2. The gene table also keeps T_spatial and
LR_spec (added for notebook 72; nothing else changes).

In [ ]:
from scipy.stats import false_discovery_control

from pseudospace.pathway_pipelines import camera_pr, moderated_f

STAT, TESTS_N, N_CALLS = {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
SQUEEZE_ROWS = []
for ds, d in DATA.items():
    for label, model in MODELS[ds].items():
        den = DENOM[ds][label]
        df2 = int(den['df2'])
        frame = moderated_f(model.statistics.T_spatial, DF1, np.asarray(den['LR_spec'], float), df2,
                            covariate=d['log_mean_count'], robust=False)
        frame['q'] = false_discovery_control(frame.p.to_numpy())
        frame['converged'] = model.statistics.converged.to_numpy() & np.asarray(den['converged'], bool)
        frame['separated'] = model.statistics.separated.to_numpy() | np.asarray(den['separated'], bool)
        frame['T_spatial'] = model.statistics.T_spatial.to_numpy()
        frame['LR_spec'] = np.asarray(den['LR_spec'], float)
        assert np.isfinite(frame.z).all() and np.isfinite(frame.F).all()
        STAT[ds][label] = frame
        test = camera_pr(frame.z, d['sets'], inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
        assert list(test.pathway_id) == list(d['sets'])
        TESTS_N[ds][label] = test
        N_CALLS[ds][label] = set(test.pathway_id[test.call.astype(bool)])
        test.assign(partition=label).to_csv(tables / f'pathway_tests_{ds}_cameraPR_{slug(label)}.csv', index=False)
        SQUEEZE_ROWS.append({'dataset': d['title'], 'partition': SHORT[ds][label], 'df2': df2,
                             'd0 (median)': float(frame.d0.median()), 's0² (median)': float(frame.s2_prior.median()),
                             'median F': float(frame.F.median()), 'genes p ≤ 0.05': int((frame.p <= .05).sum()),
                             'genes BH ≤ 0.05': int((frame.q <= GENE_FDR).sum()), 'pathway calls': len(N_CALLS[ds][label])})
    pd.concat({SHORT[ds][label]: STAT[ds][label] for label in PARTITIONS[ds]}, axis=1).to_csv(tables / f'gene_statistics_{ds}.csv')
SQUEEZE = pd.DataFrame(SQUEEZE_ROWS)
SQUEEZE.to_csv(tables / 'moderation.csv', index=False)
display(SQUEEZE.set_index(['dataset', 'partition']).round(3))

## 4 · The clustering method

Notebook 66's method B, unchanged: the `pseudospace_reconstruction` package's stage 06 (`compare_groups`) and
stage 07 (`run_pathways`) on the shared specimen curves (`clustering_first_inputs`), with:
- `CompareParams(prior_k = 0)` (other defaults, windows 0.02), `PathwayParams(min_genes = 10, max_genes = 300)` and
  node spacing 1/29;
- each specimen curve floored at half a count over the specimen's summed mt-free library;
- the zone cuts from the package's marker rule (Slc5a12, Slc13a3, Slc16a9) on the control mice, computed on the
  full-library log-normalized matrix. **Guard (G5):** the cuts increase; whether the marker runs are clean is reported.

**Runs.** The condition split with the default null (both relabelings; its calls) and with each single-split null;
each relabeling with the fair null (the other relabeling). **One call** is a pathway with matched q < 0.05 in at
least one gene group or cluster.

In [ ]:
import pseudospace_reconstruction as prc
from pseudospace_reconstruction.compare import CompareParams, Design, compare_groups
from pseudospace_reconstruction.compare.pathways import PathwayParams, ora_by_group, run_pathways
from pseudospace_reconstruction.features.axis import zone_cuts_from_markers

from pseudospace.pathway_pipelines import FLOOR_COUNTS, clustering_first_inputs, single_split_design

COMPARE_PARAMS = CompareParams(prior_k=0.)                                       # no prior in the log ratio
PATHWAY_PARAMS = PathwayParams(min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])    # the package's cap is 150
package = Path(prc.__file__).parent
external_key = digest([code_key, *sorted((package / 'compare').glob('*.py')), *sorted((package / 'features').glob('*.py')),
                       package / 'io' / 'gene_sets.py'])
ZONES, ZONE_ROWS = {}, []
for ds, d in DATA.items():
    reference = np.flatnonzero(~d['case'])
    found = zone_cuts_from_markers(d['lognorm'][reference], d['genes'], d['position'][reference], ZONE_MARKERS)
    ZONES[ds] = np.asarray(found['cuts'], float)
    assert len(ZONES[ds]) == 2 and ZONES[ds][0] < ZONES[ds][1], ZONES[ds]                 # G5
    grid = MODELS[ds]['species'].grid
    ZONE_ROWS.append({'dataset': d['title'], 'S1|S2 cut': ZONES[ds][0], 'S2|S3 cut': ZONES[ds][1],
                      'marker runs clean': bool(found['runs_clean']), 'grid': f'[{grid[0]:.3f}, {grid[-1]:.3f}]',
                      **{f'median position of {s} units': float(np.median(d['position'][d['segment'] == s])) for s in ('S1', 'S2', 'S3')}})
ZONE_TABLE = pd.DataFrame(ZONE_ROWS).set_index('dataset')
ZONE_TABLE.to_csv(tables / 'method_b_zones.csv')
display(ZONE_TABLE.round(3))
guard_log.append('G5: zone cuts increase, both datasets; marker runs clean: ' +
                 ', '.join(f"{r['dataset']} {r['marker runs clean']}" for r in ZONE_ROWS))

EXPRESSION_COLUMN = 'expression_per_10k'
GENE_COLUMNS = ['gene', 'selected', 'shape_replicated', 'direction', 'shape_class', 'location', 'group', 'cluster', 'rms',
                'level', 'shape_rms', EXPRESSION_COLUMN]


def design_of(ds, label):
    d = DATA[ds]
    per = pd.Series(PARTITIONS[ds][label][0], index=d['specimen']).groupby(level=0).first()
    return Design.from_labels(list(per.index), list(np.where(per.eq(1), 'case', 'reference')), case_label='case',
                              reference_label='reference')


def clustering_result(ds, contrast, null):
    """The package's stage 06 on the shared curves: (design, compare_groups result, axis, fit)."""
    d = DATA[ds]
    axis, fit = clustering_first_inputs(MODELS[ds][contrast], d['position'], PARTITIONS[ds][contrast][0], d['specimen'],
                                        d['library'], zone_names=list(ZONE_MARKERS), zone_cuts=ZONES[ds],
                                        zone_short_names=ZONE_SHORT)
    design = design_of(ds, contrast)
    if null is not None:
        design = single_split_design(design, group_one(ds, null))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)     # the package's own all-NaN / inf guards
        result = compare_groups(fit, axis, design, node_spacing=NODE_SPACING, elsewhere_flags=None, params=COMPARE_PARAMS)
    return design, result, axis, fit


def run_b(ds, contrast, null):
    """The package's stage 06 (compare_groups) and stage 07 (run_pathways) on the shared curves (notebook 66)."""
    design, result, _, _ = clustering_result(ds, contrast, null)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        gt = result.gene_table
        curves = result.curves()                            # float32, as its stage 07 reads them from curves.npz
        pathways = run_pathways(curves['D'].astype(float), curves['P'].astype(float), curves['N'].astype(float),
                                curves['weights'], curves['grid'], EXT_SETS[ds], gt[EXPRESSION_COLUMN].to_numpy(),
                                gt['group'].to_numpy(), cluster_labels=gt['cluster'].to_numpy(), flagged=None,
                                groups=list(result.shape_groups.index), pair_names=[str(n) for n in curves['pair_names']],
                                split_names=[str(n) for n in curves['null_names']], pair_samples=design.pairs,
                                params=PATHWAY_PARAMS)
    genes = gt[GENE_COLUMNS].copy()
    for c in ('selected', 'shape_replicated'):
        genes[c] = genes[c].fillna(False).astype(bool)
    for c in ('direction', 'shape_class', 'location', 'group', 'cluster'):
        genes[c] = genes[c].fillna('').astype(str)
    ora = pd.concat([t.assign(list=kind) for kind, t in (('group', pathways.by_group), ('cluster', pathways.by_cluster))],
                    ignore_index=True)
    ora = ora[['list', 'group', 'n_genes', 'library', 'pathway', 'set_size', 'overlap', 'fold_matched', 'p_plain', 'p_matched',
               'q_plain', 'q_matched', 'overlap_genes']].astype({'group': str, 'library': str, 'pathway': str, 'overlap_genes': str})
    ora.insert(0, 'pathway_id', ora.library + '::' + ora.pathway)
    calls = pathways.calls
    calls = pd.DataFrame({'pathway_id': (calls.library + '::' + calls.pathway).to_numpy(), 'pathway_call': calls.pathway_call.astype(str),
                          'is_representative': calls.is_representative.astype(bool), 'level_es': calls.level_es.astype(float),
                          'disruption_z_loo': calls.disruption_z_loo.astype(float), 'coherence': calls.coherence.astype(float)})
    info = pd.DataFrame({'null_names': [' | '.join(design.null_names)], 'n_splits': [len(design.mixed_splits)]})
    return {'genes': genes, 'ora': ora, 'calls': calls, 'info': info}


def flatten(tables_):
    out = {}
    for name, frame in tables_.items():
        out[f'{name}|__columns'] = np.array(list(frame.columns), dtype=str)
        for column in frame.columns:
            values = frame[column].to_numpy()
            out[f'{name}|{column}'] = values.astype(str) if values.dtype == object else values
    return out


def unflatten(payload):
    names = sorted({k.split('|', 1)[0] for k in payload if not k.startswith('__')})   # '__seconds': the stage's run time
    return {n: pd.DataFrame({c: payload[f'{n}|{c}'] for c in payload[f'{n}|__columns'].astype(str)}) for n in names}


B, B_STAGE = {ds: {} for ds in DATASETS}, {ds: {} for ds in DATASETS}
for ds, d in DATA.items():
    condition, r1, r2 = list(PARTITIONS[ds])
    curves_key = digest({n: c for n, c in MODELS[ds]['species'].specimen_curves.items()})
    for contrast, null in [(condition, None), (condition, r1), (condition, r2), (r1, r2), (r2, r1)]:
        stage = f'method_b_{ds}_{slug(contrast)}__null_{"default" if null is None else slug(null)}'
        payload = cached(stage, lambda ds=ds, contrast=contrast, null=null: flatten(run_b(ds, contrast, null)),
                         params={'compare': repr(COMPARE_PARAMS), 'pathways': repr(PATHWAY_PARAMS),
                                 'node_spacing': NODE_SPACING, 'floor': FLOOR_COUNTS},
                         inputs={**d['input_key'], 'curves': curves_key, 'group': PARTITIONS[ds][contrast][0],
                                 'zones': ZONES[ds], 'sets': d['sets'], 'null': 'default' if null is None else group_one(ds, null)},
                         key=external_key)
        B[ds][(contrast, null)] = unflatten(payload)
        B_STAGE[ds][(contrast, null)] = stage


def b_calls(t):
    """The clustering method's calls of one run: matched ORA q < 0.05 in a gene group or a cluster."""
    sig = t['ora'][t['ora'].q_matched.astype(float) < ALPHA]
    return set(sig.pathway_id)


def b_bonferroni_p(ds, t):
    """Notebook 61/66's list-based convention: the best matched ORA p over the lists x the number of lists."""
    genes = t['genes']
    n_lists = sum(int((genes[c][genes[c].ne('')].value_counts() >= PATHWAY_PARAMS.min_group_size).sum()) for c in ('group', 'cluster'))
    best = t['ora'].groupby('pathway_id').p_matched.min().astype(float)
    return (best * max(n_lists, 1)).clip(upper=1.).reindex(list(DATA[ds]['sets'])).fillna(1.).to_numpy()


RUN_ROWS = []
for ds in DATASETS:
    for (contrast, null), t in B[ds].items():
        genes = t['genes']
        selected = genes[genes.selected.astype(bool)]
        RUN_ROWS.append({'dataset': DATA[ds]['title'], 'contrast': SHORT[ds][contrast],
                         'null': 'default (both other partitions)' if null is None else SHORT[ds][null],
                         'selected genes': len(selected), 'shape confirmed': int(genes.shape_replicated.astype(bool).sum()),
                         'up / down / crossover': '{} / {} / {}'.format(*(int(selected.direction.eq(x).sum()) for x in ('up', 'down', 'crossover'))),
                         'ORA calls': len(b_calls(t)), 'redundancy clusters': len(representatives(ds, b_calls(t))),
                         'seconds': round(SECONDS[B_STAGE[ds][(contrast, null)]], 1)})
RUN_TABLE = pd.DataFrame(RUN_ROWS)
RUN_TABLE.to_csv(tables / 'clustering_runs.csv', index=False)
display(RUN_TABLE)

## 5 · Calls and pass rules

The columns are notebook 69's:
- **calls:** pathways called for the condition split; the next column gives the redundancy clusters they touch;
- **relabeled calls:** calls under the two relabelings (clustering: with the fair null);
- **NCDR:** mean relabeled calls ÷ condition calls;
- **relabeling rule** (notebook 31): NCDR < 0.25 and each relabeling below 5% of the tested pathways;
- **calibration share:** of the pooled relabeled one-sided p-values, the share ≤ 0.05; the limit is 0.075. The
  clustering method has no p-value per pathway; its line uses the list-based convention.

The rows marked **69 ·** are read from notebook 69's method table: the "before".

**Level-dominance of the clustering method** (condition run, default null):
- **L1:** the share of selected genes without a replicated shape (`shape_class` "uniform");
- **L2:** the share of calls significant only through level-only lists ("· uniform" gene groups, clusters U0/D0);
- the package's own `pathway_call` classes among the calls.

In [ ]:
METHOD_ROWS = []
N_LABEL = 'normalized · cameraPR on z'
B_LABEL, B_SINGLE_LABEL = 'clustering · ORA, default null', 'clustering · ORA, single-split null'
for ds in DATASETS:
    d = DATA[ds]
    condition, r1, r2 = list(PARTITIONS[ds])
    tested = len(d['sets'])

    def add(label, condition_sets, relabeled_sets, p_relabeled, note=''):
        n_condition = [len(c) for c in condition_sets]
        n_relabeled = [len(r) for r in relabeled_sets]
        mean_condition = float(np.mean(n_condition))
        ncdr = np.mean(n_relabeled) / mean_condition if mean_condition else np.nan
        share = float(np.mean(np.asarray(p_relabeled, float) <= ALPHA))
        METHOD_ROWS.append({
            'dataset': d['title'], 'method': label, 'calls': ', '.join(map(str, n_condition)),
            'calls (redundancy clusters)': ', '.join(str(len(representatives(ds, c))) for c in condition_sets),
            'relabeled calls': ', '.join(map(str, n_relabeled)), 'NCDR': ncdr,
            'relabeling rule': 'pass' if mean_condition and ncdr < NCDR_MAX and max(n_relabeled) < NULL_RATE * tested else 'fail',
            'calibration share': share, 'calibration': ('pass' if share <= CALIBRATION_MAX else 'fail') + note})

    add(N_LABEL, [N_CALLS[ds][condition]], [N_CALLS[ds][r1], N_CALLS[ds][r2]],
        np.concatenate([TESTS_N[ds][r].p_one_sided.to_numpy(float) for r in (r1, r2)]))
    fair = [B[ds][(r1, r2)], B[ds][(r2, r1)]]
    p_fair = np.concatenate([b_bonferroni_p(ds, t) for t in fair])
    add(B_LABEL, [b_calls(B[ds][(condition, None)])], [b_calls(t) for t in fair], p_fair, ' (list-based convention)')
    add(B_SINGLE_LABEL, [b_calls(B[ds][(condition, r)]) for r in (r1, r2)], [b_calls(t) for t in fair], p_fair,
        ' (list-based convention)')
TABLE69 = pd.read_csv(T69 / 'tables' / 'method_table.csv')
for ds in DATASETS:
    for label in (N_LABEL, B_LABEL, B_SINGLE_LABEL):
        row = TABLE69[TABLE69.dataset.eq(DATA[ds]['title']) & TABLE69.method.eq(label)]
        assert len(row) == 1, (ds, label)
        METHOD_ROWS.append({**{k: row.iloc[0][k] for k in ('calls', 'calls (redundancy clusters)', 'relabeled calls', 'NCDR',
                                                            'relabeling rule', 'calibration share', 'calibration')},
                            'dataset': DATA[ds]['title'], 'method': '69 · ' + label})
METHOD_TABLE = pd.DataFrame(METHOD_ROWS)[['dataset', 'method', 'calls', 'calls (redundancy clusters)', 'relabeled calls', 'NCDR',
                                          'relabeling rule', 'calibration share', 'calibration']]
METHOD_TABLE.to_csv(tables / 'method_table.csv', index=False)
PASS = {}
for ds in DATASETS:
    rows = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')
    PASS[ds] = rows.loc[N_LABEL, 'relabeling rule'] == 'pass' and rows.loc[N_LABEL, 'calibration'].startswith('pass')
    print(f"{DATA[ds]['title']}: {len(DATA[ds]['sets']):,} pathways tested; the normalized method "
          f"{'passes' if PASS[ds] else 'fails'} the pass rule")
    display(rows.drop(columns='dataset').round(3))


def level_dominance(t):
    """L1, L2 and the package's pathway_call classes of the calls, from one clustering run (condition, default null)."""
    genes, ora = t['genes'], t['ora']
    selected = genes[genes.selected.astype(bool)]
    sig = ora[ora.q_matched.astype(float) < ALPHA].copy()
    level_only = ((sig.list.eq('group') & sig.group.str.contains('· uniform', regex=False))
                  | (sig.list.eq('cluster') & sig.group.isin(['U0', 'D0'])))
    per = level_only.groupby(sig.pathway_id).agg(['any', 'all'])
    classes = t['calls'].set_index('pathway_id').pathway_call.reindex(sorted(set(sig.pathway_id))).value_counts()
    return {'selected genes': len(selected), 'uniform (no replicated shape)': int(selected.shape_class.eq('uniform').sum()),
            'L1': float(selected.shape_class.eq('uniform').mean()), 'calls': len(per),
            'calls through a level-only list': int(per['any'].sum()), 'calls only through level-only lists': int(per['all'].sum()),
            'L2': float(per['all'].mean()) if len(per) else np.nan,
            'pathway_call classes of the calls': '; '.join(f'{k} {v}' for k, v in classes.items())}


DOMINANCE_ROWS = []
for ds in DATASETS:
    condition = list(PARTITIONS[ds])[0]
    for run, t in (('notebook 69', unflatten(stored(T69, f'method_b_{ds}_species__null_default'))),
                   ('notebook 71', B[ds][(condition, None)])):
        DOMINANCE_ROWS.append({'dataset': DATA[ds]['title'], 'run': run, **level_dominance(t)})
DOMINANCE = pd.DataFrame(DOMINANCE_ROWS)
DOMINANCE.to_csv(tables / 'clustering_level_dominance.csv', index=False)
display(DOMINANCE.set_index(['dataset', 'run']).round(3))

## 6 · Null drift and direction balance (diagnostics)

**d_g** is the mean of Δ_g(x) over the last 21 of the 101 grid points minus the mean over the first 21. Δ is the
natural-log rate ratio, case over reference, from the condition split; d_g > 0 means "rising in human (AKI) toward
S3". Only converged, non-separated genes enter.
- **M7** (judged on human vs mouse): the median d_g over notebook 71's own genes with q > 0.5 on the normalized
  method; |value| > 0.05 is a material change. Notebook 69: +0.001.
- **Beside it** (notebook 69's check (a) layout, now 69 → 71): the median d_g over the genes with q > 0.5 in both runs.
- **Direction balance** (notebook 69's check (b)): among genes with q ≤ 0.05, the number with d_g > 0 and d_g < 0.

The two runs sit on different coordinates, so "before" and "after" are not a controlled change here: descriptive.

In [ ]:
K_WINDOW = int(round(DRIFT_SHARE * (GRID_POINTS - 1))) + 1      # 21 of the 101 grid points at each end


def drift(delta):
    """d_g: mean delta over the last 20% of the grid minus the mean over the first 20% (genes,)."""
    delta = np.asarray(delta, float)
    return delta[:, -K_WINDOW:].mean(axis=1) - delta[:, :K_WINDOW].mean(axis=1)


DRIFT69 = pd.read_csv(T69 / 'tables' / 'gene_drift_before_after.csv')
D69, OK69, Q69, D71, OK71, Q71 = {}, {}, {}, {}, {}, {}
DRIFT_ROWS, BALANCE_ROWS = [], []
for ds, d in DATA.items():
    model = MODELS[ds]['species']
    D69[ds] = DRIFT69[DRIFT69.dataset.eq(ds)].set_index('gene')['d_g after'].reindex(GENES69[ds]).astype(float)
    OK69[ds] = pd.Series(BEFORE_MODEL[ds]['stat|converged'].astype(bool) & ~BEFORE_MODEL[ds]['stat|separated'].astype(bool),
                         index=GENES69[ds])
    Q69[ds] = pd.read_csv(T69 / 'tables' / f'gene_statistics_{ds}.csv', header=[0, 1], index_col=0)[('condition', 'q')].astype(float)
    D71[ds] = pd.Series(drift(model.delta), index=d['genes'])
    OK71[ds] = model.statistics.converged & ~model.statistics.separated
    Q71[ds] = STAT[ds]['species'].q
    both = pd.Index(d['genes']).intersection(GENES69[ds])
    null_both = OK69[ds][both] & OK71[ds][both] & Q69[ds][both].gt(NULL_Q) & Q71[ds][both].gt(NULL_Q)
    d['null_genes'] = list(both[null_both.to_numpy()])
    own71 = OK71[ds] & Q71[ds].gt(NULL_Q)
    own69 = OK69[ds] & Q69[ds].gt(NULL_Q)
    m7 = float(D71[ds][own71].median())
    judged = ds == 'human_vs_mouse'
    DRIFT_ROWS.append({'dataset': d['title'], 'genes (q > 0.5 in both runs)': len(d['null_genes']),
                       'median d_g, notebook 69 (genes q > 0.5 in both)': float(D69[ds][d['null_genes']].median()),
                       'median d_g, notebook 71 (genes q > 0.5 in both)': float(D71[ds][d['null_genes']].median()),
                       'own q > 0.5 genes: notebook 69 (n)': f"{D69[ds][own69].median():.4f} ({int(own69.sum()):,})",
                       'M7: own q > 0.5 genes, notebook 71': m7, 'M7 genes': int(own71.sum()),
                       'M7 verdict': (('no material change' if abs(m7) <= DRIFT_LIMIT else 'MATERIAL CHANGE') if judged else 'reported only')})
    for run, dd, okk, qq in (('notebook 69', D69[ds], OK69[ds], Q69[ds]), ('notebook 71', D71[ds], OK71[ds], Q71[ds])):
        sig = dd[okk & qq.le(GENE_FDR)]
        BALANCE_ROWS.append({'dataset': d['title'], 'run': run, 'significant genes (q ≤ 0.05)': len(sig),
                             'rising in case toward S3 (d_g > 0)': int((sig > 0).sum()), 'falling (d_g < 0)': int((sig < 0).sum()),
                             'share rising': float((sig > 0).mean()), 'median d_g': float(sig.median())})
DRIFT = pd.DataFrame(DRIFT_ROWS)
BALANCE = pd.DataFrame(BALANCE_ROWS)
DRIFT.to_csv(tables / 'check_a_null_drift.csv', index=False)
BALANCE.to_csv(tables / 'check_b_direction_balance.csv', index=False)
pd.concat({DATA[ds]['title']: pd.DataFrame({'d_g': D71[ds], 'converged, not separated': OK71[ds], 'q': Q71[ds]})
           for ds in DATASETS}, names=['dataset', 'gene']).to_csv(tables / 'gene_drift.csv')
display(DRIFT.set_index('dataset').round(4))
display(BALANCE.set_index(['dataset', 'run']).round(3))

## 7 · Overlap between the methods

Notebook 69's check (d): the normalized calls (cameraPR on z) against the clustering calls (default null), for the
condition split: both, normalized only, clustering only, neither, and the Jaccard index, over pathways and over
redundancy clusters ("neither" counts the clusters no call of either method touches).

**Without the 8 proteasome subunits** (Psma3, Psma4, Psmb5, Psmb6, Psmc2, Psmc6, Psmd11, Psmd14):
- **normalized:** cameraPR on the same z without the 8 genes, no squeezeVar refit; sets below 10 members dropped;
- **clustering:** its ORA (`ora_by_group`) rerun with the 8 genes outside the universe, on the gene groups and
  clusters of the condition run, unchanged. **Guard (G7):** with the full universe it reproduces the run's calls.

The rows with `run` = "notebook 69" are copied from notebook 69's table.

In [ ]:
ORA_KW = dict(n_strata=PATHWAY_PARAMS.n_strata, min_set_size=PATHWAY_PARAMS.min_genes,
              min_group_size=PATHWAY_PARAMS.min_group_size, min_overlap=PATHWAY_PARAMS.min_overlap)


def clustering_calls(genes_table, sets, universe=None):
    """The clustering method's ORA calls from a run's gene groups and clusters (the package's ora_by_group)."""
    expression = genes_table[EXPRESSION_COLUMN].to_numpy(float)
    ora = pd.concat([ora_by_group(genes_table[k].to_numpy(), sets, expression, universe=universe, **ORA_KW).assign(list=k)
                     for k in ('group', 'cluster')], ignore_index=True)
    return set((ora.library + '::' + ora.pathway)[ora.q_matched < ALPHA])


def without_proteasome(sets):
    out = {p: [g for g in v if g not in PROTEASOME] for p, v in sets.items()}
    return {p: v for p, v in out.items() if len(v) >= SET_SIZES[0]}


CALLSETS = {}
for ds, d in DATA.items():
    condition = list(PARTITIONS[ds])[0]
    psm = [g for g in PROTEASOME if g in d['genes']]
    genes_table = B[ds][(condition, None)]['genes']
    assert list(genes_table.gene.astype(str)) == d['genes']
    assert clustering_calls(genes_table, EXT_SETS[ds]) == b_calls(B[ds][(condition, None)]), 'ORA does not reproduce the run'
    camera = camera_pr(STAT[ds]['species'].z.drop(psm), without_proteasome(d['sets']), inter_gene_cor=CAMERA_COR, fdr=ALPHA)
    CALLSETS[(ds, 'all genes')] = (N_CALLS[ds][condition], b_calls(B[ds][(condition, None)]))
    CALLSETS[(ds, 'without the 8 proteasome subunits')] = (
        set(camera.pathway_id[camera.call]), clustering_calls(genes_table, EXT_SETS[ds], universe=~np.isin(d['genes'], psm)))
    guard_log.append(f"{d['title']}: G7 the clustering ORA recomputed from the gene groups reproduces the run's calls; "
                     f"{len(psm)} of the 8 proteasome subunits are tested genes")

OVERLAP_ROWS = []
for (ds, universe), (n_calls, b_calls_) in CALLSETS.items():
    all_clusters = set(REDUNDANCY[ds].representative)
    for unit, x, y, total in (('pathways', n_calls, b_calls_, set(DATA[ds]['sets'])),
                              ('redundancy clusters', representatives(ds, n_calls), representatives(ds, b_calls_), all_clusters)):
        OVERLAP_ROWS.append({'dataset': DATA[ds]['title'], 'run': 'notebook 71', 'universe': universe, 'unit': unit,
                             'normalized calls': len(x), 'clustering calls': len(y), 'both': len(x & y),
                             'normalized only': len(x - y), 'clustering only': len(y - x), 'neither': len(total - (x | y)),
                             'Jaccard': len(x & y) / len(x | y) if x | y else np.nan})
OVERLAP69 = pd.read_csv(T69 / 'tables' / 'check_d_overlap.csv')
OVERLAP = pd.concat([pd.DataFrame(OVERLAP_ROWS), OVERLAP69[OVERLAP69.run.eq('notebook 69')]], ignore_index=True)
OVERLAP.to_csv(tables / 'check_d_overlap.csv', index=False)
display(OVERLAP.set_index(['dataset', 'universe', 'unit', 'run']).sort_index(level=[0, 1, 2], sort_remaining=False).round(3))
print('\n'.join('· ' + line for line in guard_log[-2:]))

## 8 · The clustering lists (pre-specified)

Notebook 69's post hoc section 8.1, now fixed in advance. For each condition run of the clustering method (default
null and each single-split null), notebook 69 (its cache) and notebook 71:
- the calls, and the list (gene group or shape cluster) that carries most of them, with its genes and calls;
- the shape clusters that hold the 8 proteasome subunits;
- the calls through the gene group "Down · gradient toward S3".

In [ ]:
LIST_ROWS = []
for ds in DATASETS:
    condition, r1, r2 = list(PARTITIONS[ds])
    for null in (None, r1, r2):
        stage = f'method_b_{ds}_{slug(condition)}__null_{"default" if null is None else slug(null)}'
        for run, t in (('notebook 69', unflatten(stored(T69, stage))), ('notebook 71', B[ds][(condition, null)])):
            genes, ora = t['genes'], t['ora']
            sig = ora[ora.q_matched.astype(float) < ALPHA]
            per = sig.groupby(['list', 'group']).pathway_id.nunique().sort_values(ascending=False)
            if len(per):
                (kind, name), top = per.index[0], int(per.iloc[0])
                carrier = f'{kind}: {name} ({int(genes[kind].eq(name).sum())} genes)'
            else:
                carrier, top = 'no call', 0
            holders = genes.set_index('gene').reindex(PROTEASOME).cluster.fillna('not tested').replace('', 'not selected').value_counts()
            LIST_ROWS.append({'dataset': DATA[ds]['title'], 'null': 'default' if null is None else SHORT[ds][null], 'run': run,
                              'calls': sig.pathway_id.nunique(), 'list carrying most calls': carrier, 'its calls': top,
                              'clusters holding the 8 subunits': ', '.join(f'{c} ({n})' for c, n in holders.items()),
                              'calls through "Down · gradient toward S3"': int(per.get(('group', 'Down · gradient toward S3'), 0))})
LISTS = pd.DataFrame(LIST_ROWS)
LISTS.to_csv(tables / 'posthoc_clustering_lists.csv', index=False)
with pd.option_context('display.max_colwidth', 50):
    display(LISTS.set_index(['dataset', 'null', 'run']))

## 9 · Comparison with notebook 69 (descriptive)

### 9.1 · Units: the old → new matching

`unit_matching_{ds}.csv` has one row per package unit (exported for later notebooks). `old_structure_id` is notebook
70's match (polygon IoU ≥ 0.5, mutual best): for human vs mouse the `structure_id` of `primary_pt_coordinate.csv`
(SCF13), for AKI the obs name in `mouse_only_v5`. Notebook 69's analysed set and positions are rebuilt by its own
rules: SCF13 as `pt_inputs` scaled it (`rescale='unit'`), the AKI DPT min–max over 69's support; both trimmed to the
common support (`package_inputs.support_positions`, checked against notebook 69 in phase 1).

Per specimen: the share matched, the Spearman of the positions on units in both sets, the 3 × 3 agreement of the
segment labels on matched units, and the units in only one set.

In [ ]:
from scipy.stats import spearmanr

from pseudospace.coordinate_inputs import read_coordinate
from pseudospace.package_inputs import support_positions


def old_analysed_set(ds):
    """Notebook 69's analysed structures: id -> (position, segment), by 69's own rules."""
    if ds == 'human_vs_mouse':
        saved = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
        obs = saved.obs[['sample', 'segment_class', 'broad_tubule_marker_call']].copy()
        saved.file.close()
        obs = obs[obs.broad_tubule_marker_call.astype(str).eq('PT')]
        coordinate = read_coordinate(coordinate_file).reindex(obs.index.astype(str))
        obs = obs[coordinate.notna().to_numpy()]
        raw, rule = coordinate.dropna().to_numpy(float), 'unit'
    else:
        saved = ad.read_h5ad(mouse_v5_file, backed='r')
        obs = saved.obs[['sample', 'segment_class', 'broad_tubule_marker_call', 'pt_subset_scanpy_dpt']].copy()
        saved.file.close()
        obs = obs[obs.broad_tubule_marker_call.astype(str).eq('PT') & obs.segment_class.astype(str).isin(['PT-S1', 'PT-S2', 'PT-S3'])]
        raw, rule = obs.pt_subset_scanpy_dpt.to_numpy(float), 'support'
    mask, position = support_positions(raw, obs['sample'].astype(str).to_numpy(), obs.segment_class.astype(str).to_numpy(), rescale=rule)
    out = obs[mask].copy()
    out['position'] = position
    return out


OLD = {ds: old_analysed_set(ds) for ds in DATASETS}
assert {ds: len(v) for ds, v in OLD.items()} == {'human_vs_mouse': 12272, 'aki_vs_control': 16991}, {ds: len(v) for ds, v in OLD.items()}
MATCH, UNIT_ROWS, SEGMENT_TABLES = {}, [], {}
for ds, d in DATA.items():
    obs = UNITS[ds].obs
    support = pd.Series(d['position'], index=d['obs'].index)
    old_id = obs.old_structure_id.astype(object).where(obs.old_structure_id.notna(), None)
    old = OLD[ds]
    m = pd.DataFrame({'unit_id': obs.index, 'sample': obs['sample'].astype(str).to_numpy(), 'polygon_id': obs.polygon_id.astype(str).to_numpy(),
                      'old_structure_id': old_id.to_numpy(), 'in_support_71': obs.index.isin(support.index),
                      'position_71': support.reindex(obs.index).to_numpy(), 'pseudospace_raw': obs.pseudospace.to_numpy(float),
                      'segment_71': obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()})
    known = pd.Index(m.old_structure_id.dropna().astype(str))
    assert known.is_unique, 'an old structure matches two package units'
    m['in_set_69'] = m.old_structure_id.isin(old.index)
    m['old_position_69'] = old.position.reindex(m.old_structure_id.astype(str)).to_numpy()
    m['old_segment_69'] = old.segment_class.astype(str).str.replace('PT-', '', regex=False).reindex(m.old_structure_id.astype(str)).to_numpy()
    m.to_csv(tables / f'unit_matching_{ds}.csv', index=False)
    MATCH[ds] = m
    both = m.in_support_71 & m.in_set_69
    for s in sorted(m['sample'].unique()):
        x = m[m['sample'].eq(s)]
        bx = x[x.in_support_71 & x.in_set_69]
        n69 = int((old['sample'].astype(str) == s).sum())
        UNIT_ROWS.append({'dataset': d['title'], 'specimen': s, 'package units': len(x), 'in 71 support': int(x.in_support_71.sum()),
                          'matched to an old structure': float(x.old_structure_id.notna().mean()),
                          'in both analysed sets': len(bx), 'notebook 69 set': n69,
                          'Spearman position (both sets)': float(spearmanr(bx.position_71, bx.old_position_69)[0]),
                          'segment labels identical (both sets)': float((bx.segment_71 == bx.old_segment_69).mean()),
                          'in 71 only': int((x.in_support_71 & ~x.in_set_69).sum()), 'in 69 only': n69 - len(bx)})
    SEGMENT_TABLES[d['title']] = pd.crosstab(m.loc[both, 'old_segment_69'], m.loc[both, 'segment_71'],
                                             rownames=['notebook 69'], colnames=['notebook 71'])
UNIT_TABLE = pd.DataFrame(UNIT_ROWS)
UNIT_TABLE.to_csv(tables / 'compare69_units.csv', index=False)
pd.concat(SEGMENT_TABLES).to_csv(tables / 'compare69_segment_agreement.csv')
display(UNIT_TABLE.set_index(['dataset', 'specimen']).round(3))
for title, table in SEGMENT_TABLES.items():
    print(title)
    display(table)

### 9.2 · Genes and pathways

**Genes** tested in both runs, per partition: the Spearman of z and T_spatial (and of d_g for the condition split),
the Jaccard of the gene sets with q ≤ 0.05, and the genes tested in only one run.

**Pathways** tested in both runs (sets tested in only one run are listed separately). Notebook 69's sets are rebuilt
by its own calls (`pt_inputs` on SCF13; its AKI helper on notebook 12's universe; guard: ids and sizes equal its saved
tables). For each method:
- kept, lost and gained calls, each with its FDR (normalized) or best matched q (clustering) before and after, and the
  median d_g of its tested members (converged, not separated) before and after (notebook 69's check (e) layout);
- the Jaccard over pathways and over redundancy clusters, using 71's clusters and, as a check, 69's;
- the Spearman of the cameraPR p (normalized method).

In [ ]:
def pathway_sets(universe, tested=None):
    """Notebook 37's pathways: members among the universe, 10-300 of them tested (notebook 69's helper)."""
    tested = list(universe) if tested is None else tested
    coverage = pd.concat([build_pathway_membership(
        json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe,
        ortholog_map=orthologs, library_name=lib, tested=tested, min_genes=1) for lib in LIBRARIES], ignore_index=True)
    coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
    coverage = coverage[coverage.n_tested.between(SET_SIZES[0], min(SET_SIZES[1], len(tested) - 1))]
    return {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}


def restrict(sets, genes):
    keep = set(genes)
    sets = {p: [g for g in v if g in keep] for p, v in sets.items()}
    return {p: v for p, v in sets.items() if SET_SIZES[0] <= len(v) <= min(SET_SIZES[1], len(keep) - 1)}


from pseudospace.coordinate_inputs import pt_inputs

nb12 = [g for g in pd.read_csv(nb12_universe_file, index_col=0).index.astype(str) if g not in MT]
inputs69 = pt_inputs(results_root, data_root, read_coordinate(coordinate_file), exclude_genes=MT)   # notebook 69's own call
assert list(inputs69['genes']) == GENES69['human_vs_mouse']
SETS69 = {'human_vs_mouse': inputs69['gene_sets'], 'aki_vs_control': restrict(pathway_sets(nb12), GENES69['aki_vs_control'])}
del inputs69
TESTS69 = {ds: pd.read_csv(T69 / 'tables' / f'pathway_tests_{ds}_cameraPR_species.csv').set_index('pathway_id') for ds in DATASETS}
RED69 = {ds: pd.read_csv(T69 / 'tables' / f'redundancy_clusters_{ds}.csv', index_col='pathway_id') for ds in DATASETS}
for ds in DATASETS:
    assert set(SETS69[ds]) == set(TESTS69[ds].index) == set(RED69[ds].index), ds
    assert all(len(SETS69[ds][p]) == int(TESTS69[ds].loc[p, 'n']) for p in SETS69[ds]), ds
guard_log.append("notebook 69's gene sets rebuilt by its own calls equal its saved tables (ids and sizes), both datasets")

GENE_ROWS = []
for ds, d in DATA.items():
    g69 = pd.read_csv(T69 / 'tables' / f'gene_statistics_{ds}.csv', header=[0, 1], index_col=0)
    both = pd.Index(d['genes']).intersection(GENES69[ds])
    for label in PARTITIONS[ds]:
        part = SHORT[ds][label]
        model69 = stored(T69, f'gene_model_{ds}_{slug(label)}')
        t69 = pd.Series(model69['stat|T_spatial'], index=GENES69[ds])
        z69, z71 = g69[(part, 'z')].astype(float), STAT[ds][label].z
        q69, q71 = g69[(part, 'q')].astype(float) <= GENE_FDR, STAT[ds][label].q <= GENE_FDR
        s69, s71 = set(both[q69[both].to_numpy()]), set(both[q71[both].to_numpy()])
        GENE_ROWS.append({'dataset': d['title'], 'partition': part, 'genes in both runs': len(both),
                          'only in 71': len(set(d['genes']) - set(GENES69[ds])), 'only in 69': len(set(GENES69[ds]) - set(d['genes'])),
                          'Spearman z': float(spearmanr(z69[both], z71[both])[0]),
                          'Spearman T_spatial': float(spearmanr(t69[both], STAT[ds][label].T_spatial[both])[0]),
                          'Spearman d_g': float(spearmanr(D69[ds][both], D71[ds][both], nan_policy='omit')[0]) if label == 'species' else np.nan,
                          'genes q ≤ 0.05: 69 / 71 / both': f'{len(s69)} / {len(s71)} / {len(s69 & s71)}',
                          'Jaccard of q ≤ 0.05 genes': len(s69 & s71) / len(s69 | s71) if s69 | s71 else np.nan})
GENE_COMPARE = pd.DataFrame(GENE_ROWS)
GENE_COMPARE.to_csv(tables / 'compare69_genes.csv', index=False)
display(GENE_COMPARE.set_index(['dataset', 'partition']).round(3))


def member_drift(values, ok, members):
    members = [g for g in members if g in values.index and bool(ok.get(g, False))]
    return float(values[members].median()) if members else np.nan


def best_q(t):
    return t['ora'].astype({'q_matched': float}).groupby('pathway_id').q_matched.min()


CHANGE_ROWS, CHANGE_SUMMARY, ONE_RUN_ROWS, CALLS69 = [], [], [], {}
for ds, d in DATA.items():
    condition = list(PARTITIONS[ds])[0]
    tested_both = set(d['sets']) & set(SETS69[ds])
    b69 = unflatten(stored(T69, f'method_b_{ds}_species__null_default'))
    calls69 = {'normalized': set(TESTS69[ds].index[TESTS69[ds].call.astype(bool)]), 'clustering': b_calls(b69)}
    calls71 = {'normalized': N_CALLS[ds][condition], 'clustering': b_calls(B[ds][(condition, None)])}
    CALLS69[ds] = calls69
    stat69 = {'normalized': TESTS69[ds].fdr, 'clustering': best_q(b69)}
    stat71 = {'normalized': TESTS_N[ds][condition].set_index('pathway_id').fdr, 'clustering': best_q(B[ds][(condition, None)])}
    for run, own, other, calls in (('notebook 69 only', SETS69[ds], d['sets'], calls69), ('notebook 71 only', d['sets'], SETS69[ds], calls71)):
        for p in sorted(set(own) - set(other)):
            ONE_RUN_ROWS.append({'dataset': d['title'], 'tested in': run, 'pathway_id': p, 'members': len(own[p]),
                                 'normalized call': p in calls['normalized'], 'clustering call': p in calls['clustering']})
    for method, stat_name in (('normalized', 'FDR'), ('clustering', 'best q')):
        before, after = calls69[method] & tested_both, calls71[method] & tested_both
        for change, members in (('kept', before & after), ('lost', before - after), ('gained', after - before)):
            for p in sorted(members):
                CHANGE_ROWS.append({'dataset': d['title'], 'method': method, 'change': change, 'pathway_id': p, 'pathway': NAMES[ds][p],
                                    'genes (71)': len(d['sets'][p]), f'{stat_name} 69': stat69[method].get(p, np.nan),
                                    f'{stat_name} 71': stat71[method].get(p, np.nan),
                                    'median d_g 69': member_drift(D69[ds], OK69[ds], SETS69[ds][p]),
                                    'median d_g 71': member_drift(D71[ds], OK71[ds], d['sets'][p])})
        rep71 = lambda calls: set(REDUNDANCY[ds].representative.reindex(sorted(calls)))   # noqa: E731
        rep69 = lambda calls: set(RED69[ds].representative.reindex(sorted(calls)))        # noqa: E731
        row = {'dataset': d['title'], 'method': method, 'sets tested in both': len(tested_both),
               'calls 69 (all)': len(calls69[method]), 'calls 71 (all)': len(calls71[method]),
               'calls 69 (tested in both)': len(before), 'calls 71 (tested in both)': len(after),
               'kept': len(before & after), 'lost': len(before - after), 'gained': len(after - before),
               'Jaccard (pathways)': len(before & after) / len(before | after) if before | after else np.nan,
               'Jaccard (redundancy clusters, 71)': (len(rep71(before) & rep71(after)) / len(rep71(before) | rep71(after))
                                                     if before | after else np.nan),
               'Jaccard (redundancy clusters, 69)': (len(rep69(before) & rep69(after)) / len(rep69(before) | rep69(after))
                                                     if before | after else np.nan)}
        if method == 'normalized':
            common = sorted(tested_both)
            row['Spearman cameraPR p'] = float(spearmanr(TESTS69[ds].p.reindex(common),
                                                         TESTS_N[ds][condition].set_index('pathway_id').p.reindex(common))[0])
        CHANGE_SUMMARY.append(row)
CHANGES = pd.DataFrame(CHANGE_ROWS)
SUMMARY_E = pd.DataFrame(CHANGE_SUMMARY)
ONE_RUN = pd.DataFrame(ONE_RUN_ROWS)
for (ds_title, method), part in CHANGES.groupby(['dataset', 'method'], sort=False):
    ds = next(k for k in DATASETS if DATA[k]['title'] == ds_title)
    part.dropna(axis=1, how='all').to_csv(tables / f'compare69_pathways_{method}_{ds}.csv', index=False)
SUMMARY_E.to_csv(tables / 'compare69_pathways_summary.csv', index=False)
ONE_RUN.to_csv(tables / 'compare69_sets_one_run.csv', index=False)
display(SUMMARY_E.set_index(['dataset', 'method']).round(3))
display(ONE_RUN.groupby(['dataset', 'tested in']).agg(sets=('pathway_id', 'size'), normalized_calls=('normalized call', 'sum'),
                                                      clustering_calls=('clustering call', 'sum')))
for (ds_title, method), part in CHANGES[CHANGES.change.ne('kept')].groupby(['dataset', 'method'], sort=False):
    print(f'\n{ds_title} · {method}: lost {int(part.change.eq("lost").sum())}, gained {int(part.change.eq("gained").sum())}')
    with pd.option_context('display.max_colwidth', 60, 'display.max_rows', 60):
        display(part.drop(columns=['dataset', 'method', 'pathway_id']).dropna(axis=1, how='all')
                .sort_values(['change', 'pathway']).head(40).set_index('pathway').round(3))

### 9.3 · Pass rules and level-dominance, 69 → 71

In [ ]:
PASS_ROWS = []
for ds in DATASETS:
    rows = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')
    for label in (N_LABEL, B_LABEL, B_SINGLE_LABEL):
        for run, key in (('notebook 69', '69 · ' + label), ('notebook 71', label)):
            r = rows.loc[key]
            PASS_ROWS.append({'dataset': DATA[ds]['title'], 'method': label, 'run': run, 'calls': r['calls'],
                              'relabeled calls': r['relabeled calls'], 'NCDR': float(r['NCDR']), 'relabeling rule': r['relabeling rule'],
                              'calibration share': float(r['calibration share']), 'calibration': r['calibration']})
    for r in DOMINANCE[DOMINANCE.dataset.eq(DATA[ds]['title'])].itertuples():
        PASS_ROWS.append({'dataset': DATA[ds]['title'], 'method': 'clustering · level-dominance', 'run': r.run,
                          'calls': r.calls, 'L1': r.L1, 'L2': r.L2})
PASS_COMPARE = pd.DataFrame(PASS_ROWS)
PASS_COMPARE.to_csv(tables / 'compare69_pass_rules.csv', index=False)
display(PASS_COMPARE.set_index(['dataset', 'method', 'run']).round(3))

## 10 · Material-change verdicts (protocol section 7)

- **M1** the positional F-test fails its pass rule on human vs mouse.
- **M2** the AKI verdict of the positional F-test changes (69: relabeling rule pass, calibration fail).
- **M3** the clustering method loses fair-null specificity on either dataset.
- **M4** the clustering method stops being level-dominated: L1 < 0.5 on human vs mouse.
- **M5** the methods start to agree: Jaccard over redundancy clusters > 0.2 on either dataset.
- **M6** the human vs mouse normalized call list moves: fewer than half of 69's called redundancy clusters (69's
  clusters, sets tested in both runs) are called again, or one of 69's five strongest calls is lost.
- **M7** the offset tilt returns: |median d_g of 71's own q > 0.5 genes| > 0.05 on human vs mouse.

In [ ]:
def rule_row(ds, label):
    return METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title']) & METHOD_TABLE.method.eq(label)].iloc[0]


hvm, aki = 'human_vs_mouse', 'aki_vs_control'
M_ROWS = []
r = rule_row(hvm, N_LABEL)
M_ROWS.append({'rule': 'M1', 'what': 'positional F-test pass rule, human vs mouse',
               'notebook 69': '97 calls; relabeled 0, 0; calibration 0.044; passes',
               'notebook 71': f"{r['calls']} calls; relabeled {r['relabeled calls']}; NCDR {r['NCDR']:.3f}; calibration {r['calibration share']:.3f}",
               'material change': not PASS[hvm]})
r69, r71 = rule_row(aki, '69 · ' + N_LABEL), rule_row(aki, N_LABEL)
verdict69 = (r69['relabeling rule'], r69['calibration'].split()[0])
verdict71 = (r71['relabeling rule'], r71['calibration'].split()[0])
M_ROWS.append({'rule': 'M2', 'what': 'positional F-test verdict, AKI (relabeling rule, calibration)',
               'notebook 69': f'{verdict69} (NCDR {float(r69["NCDR"]):.2f}, share {float(r69["calibration share"]):.3f})',
               'notebook 71': f"{verdict71} (calls {r71['calls']}; relabeled {r71['relabeled calls']}; NCDR {r71['NCDR']:.2f}; share {r71['calibration share']:.3f})",
               'material change': verdict69 != verdict71})
fails = {ds: rule_row(ds, B_LABEL)['relabeling rule'] for ds in DATASETS}
M_ROWS.append({'rule': 'M3', 'what': 'clustering fair-null relabeling rule',
               'notebook 69': 'pass, pass',
               'notebook 71': ', '.join(f"{DATA[ds]['title']}: {fails[ds]} (relabeled {rule_row(ds, B_LABEL)['relabeled calls']})" for ds in DATASETS),
               'material change': any(v != 'pass' for v in fails.values())})
dom = DOMINANCE.set_index(['dataset', 'run'])
L1 = {ds: float(dom.loc[(DATA[ds]['title'], 'notebook 71'), 'L1']) for ds in DATASETS}
L2 = {ds: float(dom.loc[(DATA[ds]['title'], 'notebook 71'), 'L2']) for ds in DATASETS}
M_ROWS.append({'rule': 'M4', 'what': 'clustering level-dominance L1, human vs mouse (limit 0.5)',
               'notebook 69': 'L1 0.704 (AKI 0.549); L2 0.083 (AKI 0.431)',
               'notebook 71': f"L1 {L1[hvm]:.3f} (AKI {L1[aki]:.3f}); L2 {L2[hvm]:.3f} (AKI {L2[aki]:.3f})",
               'material change': L1[hvm] < L1_LIMIT})
jac = OVERLAP[OVERLAP.run.eq('notebook 71') & OVERLAP.unit.eq('redundancy clusters') & OVERLAP.universe.eq('all genes')].set_index('dataset').Jaccard
M_ROWS.append({'rule': 'M5', 'what': 'Jaccard over redundancy clusters between the methods (limit 0.2)',
               'notebook 69': '0.175 (human vs mouse), 0.096 (AKI)',
               'notebook 71': ', '.join(f'{k}: {v:.3f}' for k, v in jac.items()),
               'material change': bool((jac > OVERLAP_LIMIT).any())})
tested_both = set(DATA[hvm]['sets']) & set(SETS69[hvm])
calls69_both = CALLS69[hvm]['normalized'] & tested_both
calls71_both = N_CALLS[hvm]['species'] & tested_both
clusters69 = set(RED69[hvm].representative.reindex(sorted(calls69_both)))
again = set(RED69[hvm].representative.reindex(sorted(calls71_both))) & clusters69
share_again = len(again) / len(clusters69)
top5 = {p: ('called' if p in N_CALLS[hvm]['species'] else 'not tested' if p not in DATA[hvm]['sets'] else 'lost') for p in TOP5_69}
M_ROWS.append({'rule': 'M6', 'what': "human vs mouse normalized calls: 69's called redundancy clusters called again; 69's five strongest calls",
               'notebook 69': f'{len(clusters69)} clusters called (sets tested in both)',
               'notebook 71': f"{len(again)} of {len(clusters69)} called again ({share_again:.2f}); five strongest: " +
                              '; '.join(f"{p.split('::', 1)[1]}: {v}" for p, v in top5.items()),
               'material change': share_again < M6_SHARE or any(v != 'called' for v in top5.values())})
m7 = float(DRIFT.set_index('dataset').loc[DATA[hvm]['title'], 'M7: own q > 0.5 genes, notebook 71'])
M_ROWS.append({'rule': 'M7', 'what': "median d_g of 71's own q > 0.5 genes, human vs mouse (limit |0.05|)",
               'notebook 69': '+0.001', 'notebook 71': f'{m7:+.4f}', 'material change': abs(m7) > DRIFT_LIMIT})
MATERIAL = pd.DataFrame(M_ROWS)
MATERIAL.to_csv(tables / 'material_change.csv', index=False)
with pd.option_context('display.max_colwidth', 160):
    display(MATERIAL.set_index('rule'))

## 11 · The figure

**Panels a and b: the null drift.** For the genes with q > 0.5 in both runs, the median over genes of Δ_g(x) minus
that gene's mean over the first 20% of the grid, along the PT. Dashed: notebook 69 (SCF13 or the mouse-only DPT, on
its own grid); solid: notebook 71 (the package pseudospace, on its own grid). Each run is drawn on its own position
scale; the shaded bands are notebook 71's two 20% windows; the dotted lines mark ±0.05.

**Panels c and d: overlap between the methods** (condition split, all genes), as a 2 × 2 count of pathways: rows say
whether the normalized method calls the pathway, columns whether the clustering method does. Each cell gives the
count in notebook 71 and, in brackets, notebook 69.

In [ ]:
fig = plt.figure(figsize=(180 * MM, 124 * MM))
layout = fig.add_gridspec(2, 2, height_ratios=[1.1, 1], hspace=.62, wspace=.4)
for k, (ds, tag) in enumerate(zip(DATASETS, 'ab')):
    ax = fig.add_subplot(layout[0, k])
    d, grid = DATA[ds], MODELS[ds]['species'].grid
    genes = d['null_genes']
    drift_row = DRIFT.set_index('dataset').loc[d['title']]
    for run, delta, index, x, color, ls in (('69', BEFORE_MODEL[ds]['delta'], GENES69[ds], BEFORE_MODEL[ds]['grid'], BEFORE_COLOR, '--'),
                                            ('71', MODELS[ds]['species'].delta, d['genes'], grid, AFTER_COLOR, '-')):
        rows = pd.Index(index).get_indexer(genes)
        values = delta[rows] - delta[rows, :K_WINDOW].mean(axis=1, keepdims=True)
        curve = np.median(values, axis=0)
        ax.plot(x, curve, color=color, ls=ls, lw=1.2)
        value = drift_row[f'median d_g, notebook {run} (genes q > 0.5 in both)']
        ax.text(x[-1] + .015, curve[-1], f'{run}: {value:+.3f}', color=INK, fontsize=5.4, va='center')
    for lo, hi in ((grid[0], grid[K_WINDOW - 1]), (grid[-K_WINDOW], grid[-1])):
        ax.axvspan(lo, hi, color=LIGHT, alpha=.45, lw=0, zorder=0)
    ax.axhline(0, color=LIGHT, lw=.6, zorder=0)
    for y in (-DRIFT_LIMIT, DRIFT_LIMIT):
        ax.plot([grid[-K_WINDOW], grid[-1]], [y, y], color=MUTED, lw=.6, ls=':')
    ax.text(.03, .03, f"M7 (71's own q > 0.5 genes): {drift_row['M7: own q > 0.5 genes, notebook 71']:+.3f}\n{drift_row['M7 verdict']}",
            transform=ax.transAxes, fontsize=5.4, color=MUTED, va='bottom')
    ax.set_title(f"{tag}  {d['title']}: null drift\n{len(genes):,} genes with q > 0.5 in both runs", loc='left')
    ax.set_xlabel('position along the PT (S1 → S3), each run on its own scale')
    ax.set_ylabel(f"median Δ(x) − its S1-end mean\n(ln, {d['case_name']}/{d['reference_name']})")
    ax.set_xlim(0, 1.3)
OVERLAP_INDEXED = OVERLAP.set_index(['dataset', 'run', 'universe', 'unit'])
for k, (ds, tag) in enumerate(zip(DATASETS, 'cd')):
    ax = fig.add_subplot(layout[1, k])
    after = OVERLAP_INDEXED.loc[(DATA[ds]['title'], 'notebook 71', 'all genes', 'pathways')]
    before = OVERLAP_INDEXED.loc[(DATA[ds]['title'], 'notebook 69', 'all genes', 'pathways')]
    cells = {(0, 0): 'both', (0, 1): 'normalized only', (1, 0): 'clustering only', (1, 1): 'neither'}
    for (r_, c), key in cells.items():
        ax.add_patch(plt.Rectangle((c, 1 - r_), 1, 1, facecolor='#e2ece6' if key == 'both' else '#f0efec', edgecolor='white', lw=2))
        ax.text(c + .5, 1.5 - r_, f"{int(after[key]):,}\n({int(before[key]):,})", ha='center', va='center', fontsize=6.5, color=INK)
    ax.set_xlim(0, 2)
    ax.set_ylim(0, 2)
    ax.set_aspect('equal')
    ax.set_xticks([.5, 1.5], ['called', 'not called'])
    ax.set_yticks([1.5, .5], ['called', 'not called'])
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xlabel('clustering', color=B_COLOR, fontweight='bold')
    ax.set_ylabel('normalized', color=N_COLOR, fontweight='bold')
    ax.xaxis.set_label_position('top')
    ax.set_title(f"{tag}  {DATA[ds]['title']}: pathways called\nJaccard {after['Jaccard']:.2f} (notebook 69 {before['Jaccard']:.2f})",
                 loc='left', pad=16)
fig.text(.5, .01, 'Panels c and d: count in notebook 71 (in brackets: notebook 69, on SCF13 / the mouse-only DPT).',
         ha='center', fontsize=5.4, color=MUTED)
save(fig, 'fig1_null_drift_and_overlap')
plt.show()

## 12 · Secondary sections (pre-specified; protocol section 6)

Each section is defined exactly as the comparison page's post hoc script defined it (the protocol gives the
definitions; the scripts are untracked session scratch, named there for provenance). They are all **secondary**:
the primary stays the positional F-test with cameraPR, and level stays descriptive. Inputs are notebook 71's own
objects throughout.

**Shared pass rule for every swap arm:** real calls > 0, NCDR < 0.25, each swap calls < 5% of the tested sets, and
a calibration share ≤ 0.075 (the share of swap p ≤ 0.05 pooled over both swaps). The swaps are the two
`partition_groups` relabelings.

Order: S1–S6 and S10 first; the long atlas and winsorizing sections (S7–S9) last.

### S1 · Level and total gene z, with cameraPR and GSEA prerank arms

- **Offset:** log(library) + log(sf_specimen); sf = exp(median_g[log rate_sg − mean_s log rate_sg]) over genes with
  a pseudobulk > 0 in all 4 specimens (rate = summed counts / summed library).
- **Designs:** `contrast_designs`; `nospec` = M_full without the within-group specimen contrasts (a relabeling adds
  the 0/1 species column). α: the condition split's.
- **LRs** (`nested_nb_lr`): T_level = base → level; T_total = base → full; LR_spec_level = nospec → full (df 2 for the
  condition split, 1 for a relabeling). **Sign:** the group coefficient of the M_level fit.
- **Level z:** `moderated_f(T_level, 1, LR_spec_level, df_lvl)`, signed z = sign · Φ⁻¹(1 − p/2). **Total z:**
  `moderated_f(T_total, 7, LR_spec_level + LR_spec, df_lvl + df₂)`, the upper-tail z. Both with covariate log mean
  count. **Shape z:** the primary z.
- **Set tests:** cameraPR (0.01): level calls FDR ≤ 0.05 either direction, calibration p two-sided; shape and total
  Up and FDR ≤ 0.05, calibration p one-sided. GSEA prerank (`rank_first_gsea`, raw metric, 1,000 permutations,
  seed 12, sizes 10–300): level q ≤ 0.05, p nominal; shape and total NES > 0 and q ≤ 0.05, p one-sided.

In [ ]:
import gseapy as gp
from scipy.stats import norm

from pseudospace.nb_gam import nb_fit, nested_nb_lr
from pseudospace.pathway_calibration import contrast_designs
from pseudospace.pathway_pipelines import rank_first_gsea
from pseudospace.stats_gam import gam_internal_knots, make_gam_design

P_B = BASIS_DF + 1                                   # intercept + 6 basis columns
ARMS = {'shape': 'shape (T_spatial)', 'level': 'level signed (T_level)', 'total': 'total (T_total)'}


def size_factors(d):
    names = np.unique(d['specimen'])
    pb = np.vstack([np.asarray(d['counts'][d['specimen'] == n].sum(axis=0)).ravel() for n in names])
    rate = pb / np.array([d['library'][d['specimen'] == n].sum() for n in names])[:, None]
    ok = (pb > 0).all(axis=0)
    lr_ = np.log(rate[:, ok])
    return pd.Series(np.exp(np.median(lr_ - lr_.mean(axis=0), axis=1)), index=names)


def level_lrs(ds, label, sf):
    d = DATA[ds]
    group, nuisance = PARTITIONS[ds][label]
    offset = np.log(d['library']) + np.log(sf.reindex(d['specimen']).to_numpy())
    knots = gam_internal_knots(d['position'], basis_df=BASIS_DF)
    designs, w = contrast_designs(d['position'], group, d['specimen'], knots, nuisance_shape=nuisance)
    nospec = np.delete(designs['full'], np.r_[2 * P_B:2 * P_B + 2], axis=1)
    if nuisance is not None:
        nospec = np.column_stack([nospec, np.asarray(nuisance, float)])
    designs = dict(designs, nospec=nospec)
    disp = MODELS[ds]['species'].dispersion
    lr = nested_nb_lr(d['counts'], designs, w, offset, disp, {'T_level': ('base', 'level'), 'T_spatial': ('level', 'full'),
                      'T_total': ('base', 'full'), 'LR_spec_level': ('nospec', 'full')}, block_size=200, n_jobs=N_JOBS)
    fit = nb_fit(d['counts'], designs['level'], w, offset, disp, block_size=200, n_jobs=N_JOBS)
    out = {k: np.asarray(lr[k]) for k in ('T_level', 'T_spatial', 'T_total', 'LR_spec_level', 'converged', 'separated')}
    out.update(sign=np.sign(fit['beta'][:, P_B]), df_lvl=np.array(lr['df']['LR_spec_level'][0]))
    return out


SF, LVL, Z = {}, {}, {}
for ds, d in DATA.items():
    SF[ds] = size_factors(d)
    LVL[ds], Z[ds] = {}, {}
    for label in PARTITIONS[ds]:
        LVL[ds][label] = cached(f's1_level_lr_{ds}_{slug(label)}', lambda ds=ds, label=label: level_lrs(ds, label, SF[ds]),
                                params={'basis_df': BASIS_DF, 'label': label},
                                inputs={**d['input_key'], 'sf': SF[ds].to_numpy(), 'alpha': MODELS[ds]['species'].dispersion['alpha']})
        r = LVL[ds][label]
        df2 = int(DENOM[ds][label]['df2'])
        genes = pd.Index(d['genes'])
        lv = moderated_f(pd.Series(r['T_level'], index=genes), 1, r['LR_spec_level'], int(r['df_lvl']), covariate=d['log_mean_count'])
        tt = moderated_f(pd.Series(r['T_total'], index=genes), 7, r['LR_spec_level'] + np.asarray(DENOM[ds][label]['LR_spec'], float),
                         int(r['df_lvl']) + df2, covariate=d['log_mean_count'])
        Z[ds][label] = {ARMS['shape']: STAT[ds][label].z,
                        ARMS['level']: pd.Series(r['sign'] * norm.isf(lv.p.clip(lower=1e-300) / 2), index=genes),
                        ARMS['total']: tt.z}
    assert int(LVL[ds]['species']['df_lvl']) == 2 and all(int(LVL[ds][l]['df_lvl']) == 1 for l in list(PARTITIONS[ds])[1:])
    pd.concat({SHORT[ds][label]: pd.DataFrame(Z[ds][label]) for label in PARTITIONS[ds]}, axis=1).to_csv(tables / f's1_gene_z_{ds}.csv')
pd.DataFrame({DATA[ds]['title']: SF[ds] for ds in DATASETS}).to_csv(tables / 's1_size_factors.csv')


def gsea_payload(z, sets):
    t = rank_first_gsea(z, sets, permutation_num=1000, threads=N_JOBS)
    return {'pathway_id': t.pathway_id.to_numpy().astype(str), 'NES': t.NES.to_numpy(float), 'p': t.p.to_numpy(float),
            'q': t.q.to_numpy(float), 'p_one_sided': t.p_one_sided.to_numpy(float), 'call': t.call.to_numpy(bool)}


S1_CALLS, S1_ROWS = {ds: {} for ds in DATASETS}, []
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    per_arm = []
    for arm, stat in ARMS.items():
        for test in ('cameraPR', 'GSEA prerank'):
            per = {}
            for label in labels:
                z = Z[ds][label][stat]
                if test == 'cameraPR':
                    t = camera_pr(z, d['sets'], inter_gene_cor=CAMERA_COR).set_index('pathway_id')
                    hit, p = (t.fdr.le(ALPHA), t.p) if arm == 'level' else (t.call, t.p_one_sided)
                    frame = pd.DataFrame({'hit': hit, 'p': p, 'fdr': t.fdr, 'p_two_sided': t.p, 'direction': t.direction})
                else:
                    g = cached(f's1_gsea_{ds}_{arm}_{slug(label)}', lambda z=z: gsea_payload(z, d['sets']),
                               params={'permutation_num': 1000, 'seed': 12, 'gseapy': gp.__version__, 'sizes': SET_SIZES},
                               inputs={'z': z.to_numpy(), 'genes': d['genes'], 'sets': d['sets']})
                    t = pd.DataFrame({k: g[k] for k in ('NES', 'p', 'q', 'p_one_sided', 'call')}, index=g['pathway_id'].astype(str))
                    hit, p = (t.q.le(ALPHA), t.p) if arm == 'level' else (t.call.astype(bool), t.p_one_sided)
                    frame = pd.DataFrame({'hit': hit, 'p': p, 'fdr': t.q, 'p_two_sided': t.p,
                                          'direction': np.where(t.NES > 0, 'Up', 'Down'), 'NES': t.NES})
                per[label] = frame
            S1_CALLS[ds][(arm, test)] = per
            n_true = int(per[labels[0]].hit.sum())
            n_null = [int(per[l].hit.sum()) for l in labels[1:]]
            share = float((np.concatenate([per[l].p.to_numpy(float) for l in labels[1:]]) <= ALPHA).mean())
            ncdr = np.mean(n_null) / n_true if n_true else np.nan
            tested = len(per[labels[0]])
            passes = bool(n_true) and ncdr < NCDR_MAX and all(n < NULL_RATE * tested for n in n_null) and share <= CALIBRATION_MAX
            S1_ROWS.append({'dataset': d['title'], 'arm': arm, 'statistic': stat, 'test': test, 'calls': n_true,
                            'relabeled': f'{n_null[0]}, {n_null[1]}', 'NCDR': ncdr, 'calibration share': share, 'tested': tested,
                            'pass': 'pass' if passes else 'fail'})
            per_arm.append(pd.concat({f'{arm} · {test} · {SHORT[ds][l]}': per[l] for l in labels}, axis=1))
    pd.concat(per_arm, axis=1).rename_axis('pathway_id').to_csv(tables / f's1_pathway_tests_{ds}.csv')
S1_PASS = pd.DataFrame(S1_ROWS)
S1_PASS.to_csv(tables / 's1_pass_table.csv', index=False)
display(S1_PASS.set_index(['dataset', 'arm', 'test']).round(3))
assert all((S1_CALLS[ds][('shape', 'cameraPR')][list(PARTITIONS[ds])[0]].hit.to_numpy()
            == TESTS_N[ds]['species'].call.to_numpy()).all() for ds in DATASETS), 'the shape arm must be the primary'

### S2 · ssGSEA + limma (specimen × 6-bin pseudobulk)

- **Bins:** 6, at the quantiles 1/6…5/6 of the pooled position (`searchsorted`, right).
- **Pseudobulk:** specimen × bin log2(Σcounts / Σlibrary · 10⁶ + 1) over the tested genes.
- **Scoring:** `gseapy.ssgsea(sample_norm_method='rank', min_size=10, max_size=300)`, weight 0.25; the score is the NES.
- **Level:** limma `eBayes` on the specimen-mean score, design [1, group (+ nuisance)], coefficient group.
- **Shape:** design [specimen indicators, bin dummies 1–5, group × bin (+ nuisance × bin)]; moderated F on group × bin.
- A call is BH ≤ 0.05; the swaps and the pass rule as above.

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

limma = importr('limma')
N_BINS6 = 6


def limma_test(Y, X, coefs):
    """Y sets x samples, X design (samples x p); moderated F (or t) p-values for coefs (0-based)."""
    rY = ro.r.matrix(ro.FloatVector(Y.to_numpy().ravel()), nrow=Y.shape[0], byrow=True)
    rX = ro.r.matrix(ro.FloatVector(X.T.ravel()), nrow=X.shape[0])
    fit = limma.eBayes(limma.lmFit(rY, rX))
    tt = limma.topTable(fit, coef=ro.IntVector([c + 1 for c in coefs]), number=Y.shape[0], sort_by='none')
    p, q = np.asarray(tt.rx2('P.Value')), np.asarray(tt.rx2('adj.P.Val'))
    sign = np.sign(np.asarray(tt.rx2('logFC'))) if len(coefs) == 1 else np.ones(len(p))
    return pd.DataFrame({'p': p, 'q': q, 'sign': sign}, index=Y.index)


def ssgsea_scores(ds):
    d = DATA[ds]
    cuts = np.quantile(d['position'], np.linspace(0, 1, N_BINS6 + 1)[1:-1])
    b = np.searchsorted(cuts, d['position'], side='right')
    cols = {}
    for n in np.unique(d['specimen']):
        for k in range(N_BINS6):
            m = (d['specimen'] == n) & (b == k)
            cols[f'{n}|{k}'] = np.log2(np.asarray(d['counts'][m].sum(axis=0)).ravel() / d['library'][m].sum() * 1e6 + 1)
    E = pd.DataFrame(cols, index=d['genes'])
    res = gp.ssgsea(data=E, gene_sets=d['sets'], outdir=None, sample_norm_method='rank', min_size=SET_SIZES[0],
                    max_size=SET_SIZES[1], no_plot=True, threads=N_JOBS, verbose=False)
    S = res.res2d.pivot(index='Term', columns='Name', values='NES').astype(float)[list(E.columns)]
    S = S.loc[[p for p in d['sets'] if p in S.index]]
    return {'scores': S.to_numpy(float), 'sets': S.index.to_numpy().astype(str), 'samples': S.columns.to_numpy().astype(str)}


S2_ROWS, S2_TABLES = [], {}
for ds, d in DATA.items():
    g = cached(f's2_ssgsea_{ds}', lambda ds=ds: ssgsea_scores(ds),
               params={'bins': N_BINS6, 'gseapy': gp.__version__, 'sizes': SET_SIZES, 'norm': 'rank'},
               inputs={**d['input_key'], 'sets': d['sets']})
    S = pd.DataFrame(g['scores'], index=g['sets'].astype(str), columns=g['samples'].astype(str))
    meta = pd.DataFrame([c.split('|') for c in S.columns], columns=['specimen', 'bin'], index=S.columns).astype({'bin': int})
    names = sorted(meta.specimen.unique())
    out = {}
    for label, (group, nuisance) in PARTITIONS[ds].items():
        gs = pd.Series(group, index=d['specimen']).groupby(level=0).first()
        ns = None if nuisance is None else pd.Series(nuisance, index=d['specimen']).groupby(level=0).first()
        L = S.T.groupby(meta.specimen).mean().T[names]
        XL = np.column_stack([np.ones(len(names)), gs[names].to_numpy()] + ([] if ns is None else [ns[names].to_numpy()]))
        lev = limma_test(L, XL, [1])
        sp = np.column_stack([(meta.specimen == n).to_numpy(float) for n in names])
        bins = np.column_stack([(meta.bin == k).to_numpy(float) for k in range(1, N_BINS6)])
        gcol = meta.specimen.map(gs).to_numpy(float)
        XS = [sp, bins, gcol[:, None] * bins]
        if ns is not None:
            XS.append(meta.specimen.map(ns).to_numpy(float)[:, None] * bins)
        shp = limma_test(S, np.column_stack(XS), list(range(len(names) + N_BINS6 - 1, len(names) + 2 * (N_BINS6 - 1))))
        out[label] = {'level': lev, 'shape': shp}
    labels = list(out)
    for test in ('level', 'shape'):
        n_true = int(out[labels[0]][test].q.le(ALPHA).sum())
        n_null = [int(out[l][test].q.le(ALPHA).sum()) for l in labels[1:]]
        share = float(np.mean(np.concatenate([out[l][test].p.to_numpy() for l in labels[1:]]) <= ALPHA))
        ncdr = np.mean(n_null) / n_true if n_true else np.nan
        passes = bool(n_true) and ncdr < NCDR_MAX and all(n < NULL_RATE * len(S) for n in n_null) and share <= CALIBRATION_MAX
        S2_ROWS.append({'dataset': d['title'], 'ssGSEA test': test, 'calls': n_true, 'of sets': len(S),
                        'relabeled': f'{n_null[0]}, {n_null[1]}', 'NCDR': ncdr, 'calibration share': share,
                        'pass': 'pass' if passes else 'fail'})
    S2_TABLES[ds] = pd.concat({f'{test} · {SHORT[ds][l]}': out[l][test] for l in labels for test in ('level', 'shape')}, axis=1)
    S2_TABLES[ds].rename_axis('pathway_id').to_csv(tables / f's2_ssgsea_tests_{ds}.csv')
S2_PASS = pd.DataFrame(S2_ROWS)
S2_PASS.to_csv(tables / 's2_ssgsea_pass_table.csv', index=False)
display(S2_PASS.set_index(['dataset', 'ssGSEA test']).round(3))

### S3 · Level gene list + ORA (human vs mouse only)

- **L, dh and dm** follow notebook 68's definitions on notebook 71's condition-split model: L =
  `grid_centred_level(Δ, reference = converged & not separated)`; dh, dm and the cross pairs are `pair_levels` of the
  specimen curves (valid = that specimen's fit converged and not separated, and the gene converged).
- **Kept genes:** converged, not separated, dh and dm present, and the atlas columns `A` and `A_census_human` of
  notebook 68's `gene_level.csv` present (coordinate-free). Universe genes absent from that table are dropped and
  counted.
- **Effect and variance:** condition split eff = L, s² = (dh² + dm²)/4; relabeling COR1 + a: eff = (dh ± dm)/2 (+ for
  a = Ctrl1A2), s² = (w₁² + w₂²)/4 with w₁ = "HUK1_COR1 − a", w₂ = "HUK1_MED1 − b", NaN filled with the median.
- **Moderation:** `squeezeVar(s², 2, covariate = log mean count)`; t = eff/√s²_post on 2 + d₀ df, two-sided; BH.
- **Lists:** Up and Down at q ≤ 0.05. **ORA:** hypergeometric, background the kept genes, sets restricted to them
  (≥ 2 members), BH over set × direction; the best row per set is the set's result.

In [ ]:
from scipy.stats import hypergeom
from scipy.stats import t as t_dist
from statsmodels.stats.multitest import multipletests

from pseudospace.species_level import grid_centred_level, pair_levels

d = DATA[hvm]
model = MODELS[hvm]['species']
genes = pd.Index(d['genes'])
CONVERGED, SEPARATED = model.statistics.converged.to_numpy(), model.statistics.separated.to_numpy()
REF = CONVERGED & ~SEPARATED
MOUSE, HUMAN = d['reference_specimens'], d['case_specimens']
assert MOUSE == ['Ctrl1A2', 'Ctrl1A4'] and HUMAN == ['HUK1_COR1', 'HUK1_MED1']
L_level = grid_centred_level(pd.DataFrame(model.delta, index=genes), reference=REF).to_numpy()
L_level[~CONVERGED] = np.nan
valid = {n: (model.specimen_converged[n] & ~model.specimen_separated[n]).to_numpy() & CONVERGED for n in MOUSE + HUMAN}
PAIRS_X = pair_levels(model.specimen_curves, [(h, m) for h in HUMAN for m in MOUSE], valid=valid, reference=REF, index=genes)
SAME = pair_levels(model.specimen_curves, [tuple(MOUSE), tuple(HUMAN)], valid=valid, reference=REF, index=genes)
atlas68 = pd.read_csv(nb68_tables / 'gene_level.csv', index_col=0)[['A', 'A_census_human']]
g3 = pd.DataFrame({'L': L_level, 'dh': SAME['HUK1_COR1 - HUK1_MED1'], 'dm': SAME['Ctrl1A2 - Ctrl1A4']}, index=genes).join(PAIRS_X)
keep = (REF & g3.dh.notna().to_numpy() & g3.dm.notna().to_numpy() & genes.isin(atlas68.index)
        & atlas68.reindex(genes).notna().all(axis=1).to_numpy())
S3_DROPPED = {'not converged or separated': int((~REF).sum()), 'dh or dm missing': int((REF & (g3.dh.isna() | g3.dm.isna()).to_numpy()).sum()),
              'absent from notebook 68 atlas columns': int((~genes.isin(atlas68.index) | atlas68.reindex(genes).isna().any(axis=1).to_numpy()).sum()),
              'kept': int(keep.sum()), 'universe': len(genes)}
g3 = g3[keep]
k_genes = g3.index
logmean3 = pd.Series(d['log_mean_count'], index=genes).reindex(k_genes)
idx3 = {p: [k_genes.get_loc(x) + 1 for x in sorted(set(v) & set(k_genes))] for p, v in d['sets'].items()}
idx3 = {p: v for p, v in idx3.items() if len(v) >= 2}
S3_DESIGNS = {'species': None, 'relabeled COR1+Ctrl1A2': 'Ctrl1A2', 'relabeled COR1+Ctrl1A4': 'Ctrl1A4'}
S3_ROWS, S3_ORA = [], {}
for dname, a in S3_DESIGNS.items():
    if a is None:
        eff, s2 = g3.L, (g3.dh ** 2 + g3.dm ** 2) / 4
    else:
        b = 'Ctrl1A4' if a == 'Ctrl1A2' else 'Ctrl1A2'
        eff = (g3.dh + (g3.dm if a == 'Ctrl1A2' else -g3.dm)) / 2
        w1, w2 = g3[f'HUK1_COR1 - {a}'], g3[f'HUK1_MED1 - {b}']
        s2 = (w1 ** 2 + w2 ** 2) / 4
        s2 = s2.fillna(s2.median())
    sv = limma.squeezeVar(ro.FloatVector(s2.to_numpy()), 2, covariate=ro.FloatVector(logmean3.to_numpy()))
    post, dft = np.asarray(sv.rx2('var.post')), 2 + np.asarray(sv.rx2('df.prior'))
    tt = eff.to_numpy() / np.sqrt(post)
    p = 2 * t_dist.sf(np.abs(tt), dft)
    q = multipletests(p, method='fdr_bh')[1]
    N, recs = len(k_genes), []
    for direction, sel in (('Up', (q <= .05) & (tt > 0)), ('Down', (q <= .05) & (tt < 0))):
        K = int(sel.sum())
        for pid, v in idx3.items():
            x = int(sel[np.array(v) - 1].sum())
            recs.append((pid, direction, x, len(v), hypergeom.sf(x - 1, N, K, len(v)) if K else 1.0))
    ora = pd.DataFrame(recs, columns=['pathway_id', 'direction', 'hits', 'm', 'p'])
    ora['fdr'] = multipletests(ora.p, method='fdr_bh')[1]
    S3_ORA[dname] = ora
    ora.to_csv(tables / f's3_level_ora_{dname.replace(" ", "_").replace("+", "_")}.csv', index=False)
    S3_ROWS.append({'arm': f'gene list q<=0.05 + ORA · {dname}', 'calls': int(ora[ora.fdr.le(.05)].pathway_id.nunique()),
                    'genes in list': f"{int(((q <= .05) & (tt > 0)).sum())} up / {int(((q <= .05) & (tt < 0)).sum())} down",
                    'genes kept': N})
S3_SUMMARY = pd.DataFrame(S3_ROWS).set_index('arm')
S3_SUMMARY.to_csv(tables / 's3_level_variants_summary.csv')
g3.to_csv(tables / 's3_level_gene_table.csv')
display(S3_SUMMARY)
print('S3 genes:', S3_DROPPED)
ora_species = S3_ORA['species']
S3_BEST = ora_species.sort_values('p').drop_duplicates('pathway_id').set_index('pathway_id')
S3_CALLS = set(S3_BEST.index[S3_BEST.fdr.le(.05)])

### S4–S6 · p-value histograms, implied inter-gene correlation, overlap of every arm with clustering

- **S4:** the cameraPR calibration p of each S1 arm (one-sided Up for shape and total, two-sided for level), for the
  real split and each swap, in 20 equal bins on [0, 1] (last bin closed), with the share ≤ 0.05.
- **S5:** per arm and partition, μ and σ² (ddof 1) of z over all genes; for each tested set
  y_s = n_s(z̄_s − μ)²/σ² − 1; ρ̂ = Σ(n_s − 1)y_s / Σ(n_s − 1)².
- **S6:** each arm's calls (condition split) against the clustering calls (default null): both, arm only, clustering
  only, neither, Jaccard over pathways and over redundancy clusters; for signed arms the direction agreement with
  clustering's best list (minimum q_matched, then p_matched, then larger overlap; "Up"/"U…" up, "Down"/"D…" down).

In [ ]:
EDGES20 = np.linspace(0, 1, 21)
S4_ROWS, S5_ROWS, S6_ROWS = [], [], []


def list_direction(name):
    if name.startswith(('Up', 'U')):
        return 'Up'
    if name.startswith(('Down', 'D')):
        return 'Down'
    return 'Cross'


for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    genes = pd.Index(d['genes'])
    gi = {p: genes.get_indexer(v) for p, v in d['sets'].items()}
    nn = np.array([len(gi[p]) for p in d['sets']], float)
    for arm in ARMS:
        per = S1_CALLS[ds][(arm, 'cameraPR')]
        for part, label in zip(('real', 'swap1', 'swap2'), labels):
            p = per[label].p.to_numpy(float)
            counts_, _ = np.histogram(p, bins=EDGES20)
            S4_ROWS.append({'dataset': d['title'], 'arm': f'{arm} cameraPR', 'partition': part,
                            'p_used': 'two-sided p' if arm == 'level' else 'one-sided (Up) p', 'n': len(p),
                            'counts': json.dumps(counts_.tolist()), 'share_le_05': float((p <= ALPHA).mean())})
            z = Z[ds][label][ARMS[arm]].reindex(genes).to_numpy(float)
            mu, var = z.mean(), z.var(ddof=1)
            m = np.array([z[gi[p_]].mean() for p_ in d['sets']])
            y = nn * (m - mu) ** 2 / var - 1
            S5_ROWS.append({'dataset': d['title'], 'arm': arm, 'partition': part,
                            'implied inter-gene correlation': float(np.sum((nn - 1) * y) / np.sum((nn - 1) ** 2))})
    condition = labels[0]
    ora = B[ds][(condition, None)]['ora'].copy()
    for c in ('q_matched', 'p_matched', 'overlap'):
        ora[c] = ora[c].astype(float)
    best = ora.sort_values(['q_matched', 'p_matched', 'overlap'], ascending=[True, True, False]).drop_duplicates('pathway_id').set_index('pathway_id')
    best_dir = best.group.astype(str).map(list_direction)
    clu = b_calls(B[ds][(condition, None)])
    ids = set(d['sets'])
    arms = [('shape cameraPR (positional F-test)', S1_CALLS[ds][('shape', 'cameraPR')][condition], None),
            ('level cameraPR (signed)', S1_CALLS[ds][('level', 'cameraPR')][condition], 'direction'),
            ('total cameraPR', S1_CALLS[ds][('total', 'cameraPR')][condition], None),
            ('shape GSEA prerank', S1_CALLS[ds][('shape', 'GSEA prerank')][condition], None),
            ('level GSEA prerank (signed)', S1_CALLS[ds][('level', 'GSEA prerank')][condition], 'direction'),
            ('total GSEA prerank', S1_CALLS[ds][('total', 'GSEA prerank')][condition], None)]
    if ds == hvm:
        arms.append(('level ORA form (level gene list + ORA)', pd.DataFrame({'hit': S3_BEST.fdr.le(.05), 'direction': S3_BEST.direction}), 'direction'))
    for arm, frame, signed in arms:
        mine = set(frame.index[frame.hit.astype(bool)])
        rm, rc = representatives(ds, mine), representatives(ds, clu)
        row = {'dataset': d['title'], 'arm': arm, 'arm calls': len(mine), 'clustering calls': len(clu), 'both': len(mine & clu),
               'arm only': len(mine - clu), 'clustering only': len(clu - mine), 'neither': len(ids - mine - clu),
               'Jaccard': len(mine & clu) / max(1, len(mine | clu)), 'Jaccard (redundancy clusters)': len(rm & rc) / max(1, len(rm | rc))}
        if signed:
            shared = [p for p in mine & clu if best_dir.get(p) in ('Up', 'Down')]
            row['direction agrees'] = int(sum(frame.direction[p] == best_dir[p] for p in shared))
            row['direction comparable'] = len(shared)
        S6_ROWS.append(row)
S4 = pd.DataFrame(S4_ROWS)
S5 = pd.DataFrame(S5_ROWS)
S6 = pd.DataFrame(S6_ROWS)
S4.to_csv(tables / 's4_p_histograms.csv', index=False)
S5.to_csv(tables / 's5_implied_correlation.csv', index=False)
S6.to_csv(tables / 's6_overlap_arms.csv', index=False)
display(S4.drop(columns='counts').pivot_table(index=['dataset', 'arm'], columns='partition', values='share_le_05').round(3))
display(S5.pivot_table(index=['dataset', 'arm'], columns='partition', values='implied inter-gene correlation').round(3))
display(S6.set_index(['dataset', 'arm']).round(3))

### S10 · Per-gene mechanics tables

Saved directly so the comparison page needs no refit:
- `s10_mechanics_genes_{ds}.csv`: log10 mean count; T_spatial/6, LR_spec/12, s²_post, F, z, q, fit_ok; level_lfc
  (the group coefficient of an all-gene M_level fit with the mt-free offset, ÷ ln 2); level_z from S1; and from the
  condition run's `compare_groups`: selected, direction, shape_class, location, group, cluster, rms, level, shape_rms,
  null_rms, min_pair_reproduction, shape_null_ratio, min_pair_shape_reproduction, the expression stratum, the
  whole-axis, shape and local-window floors, and the local-window statistics.
- `s10_mechanics_{ds}.npz`: the grid and clustering weights; D, P, N and S; `specimen_filled` and the floored
  specimen log2 curves; the specimen, group and Δ curves (ln); the M_level group curves (ln); the 20-bin observed
  rates per specimen; segment shares per bin; the cluster table; the zone cuts. **Guard:** the rerun of the condition
  run reproduces the cached gene table.

In [ ]:
from pseudospace_reconstruction.compare.selection import expression_strata


def level_fit_payload(ds):
    d, model = DATA[ds], MODELS[ds]['species']
    group = PARTITIONS[ds]['species'][0]
    designs, w = contrast_designs(d['position'], group, d['specimen'], model.knots)
    base = make_gam_design(model.grid, model.knots)
    rows = {}
    for level_ in (0., 1.):
        r2 = np.zeros((len(model.grid), designs['level'].shape[1]))
        r2[:, :P_B], r2[:, P_B] = base, level_
        rows[level_] = r2
    fit = nb_fit(d['counts'], designs['level'], w, np.log(d['library']), model.dispersion,
                 contrasts={'ref': rows[0.], 'case': rows[1.]}, block_size=200, n_jobs=N_JOBS)
    return {'beta_group': fit['beta'][:, P_B], 'converged': fit['converged'],
            'level_reference': fit['contrasts']['ref']['value'], 'level_case': fit['contrasts']['case']['value']}


MECH_SUMMARY = []
for ds, d in DATA.items():
    model, condition = MODELS[ds]['species'], 'species'
    lvl = cached(f's10_level_fit_{ds}', lambda ds=ds: level_fit_payload(ds),
                 params={'basis_df': BASIS_DF}, inputs={**d['input_key'], 'alpha': model.dispersion['alpha']})
    design, result, axis, fit = clustering_result(ds, condition, None)
    gt = result.gene_table
    cachedgt = B[ds][(condition, None)]['genes']
    assert (gt.selected.fillna(False).astype(bool).to_numpy() == cachedgt.selected.astype(bool).to_numpy()).all()
    assert (gt.cluster.fillna('').astype(str).to_numpy() == cachedgt.cluster.astype(str).to_numpy()).all()
    np.testing.assert_allclose(gt.rms.to_numpy(float), cachedgt.rms.to_numpy(float), rtol=0, atol=1e-9, equal_nan=True)
    frame = STAT[ds][condition]
    stratum = expression_strata(gt[EXPRESSION_COLUMN].to_numpy(), 10)
    local = result.selection.local
    table = pd.DataFrame({
        'log10_mean_count': np.log10(np.asarray(d['counts'].mean(axis=0)).ravel()),
        'level_lfc': lvl['beta_group'] / np.log(2), 'level_fit_converged': lvl['converged'],
        'level_z': Z[ds][condition][ARMS['level']].to_numpy(), 'shape_num': frame.T_spatial / DF1, 's2_raw': frame.s2,
        'shape_den': frame.s2_post, 'shape_F': frame.F, 'shape_z': frame.z, 'shape_q': frame.q, 'p': frame.p, 'd0': frame.d0,
        'df_total': frame.df_total, 's2_prior': frame.s2_prior, 'T_spatial': frame.T_spatial, 'LR_spec': frame.LR_spec,
        'T_level_gene_model': model.statistics.T_level.to_numpy(), 'alpha': model.statistics.alpha.to_numpy(),
        'fit_ok': (model.statistics.converged & ~model.statistics.separated).to_numpy(),
        **{c: gt[c].to_numpy() for c in ('selected', 'selected_whole_axis', 'selected_local_window', 'shape_replicated', 'direction',
                                         'shape_class', 'location', 'group', 'cluster', 'rms', 'level', 'shape_rms', 'null_rms',
                                         'min_pair_reproduction', 'shape_null_ratio', 'min_pair_shape_reproduction', EXPRESSION_COLUMN)},
        'stratum': stratum, 'floor_whole_axis': result.selection.floor, 'floor_shape': result.selection.shape_floor,
        **{f'local_{k}': np.asarray(local[k]) for k in ('x', 'M', 'Mnull', 'rep', 'level', 'floor')}}, index=pd.Index(d['genes'], name='gene'))
    table.to_csv(tables / f's10_mechanics_genes_{ds}.csv')
    con, sel = result.contrasts, result.selection
    grid = model.grid
    edges = np.linspace(grid[0], grid[-1], 21)
    bins_ = np.clip(np.digitize(d['position'], edges) - 1, 0, 19)
    inside = (d['position'] >= edges[0]) & (d['position'] <= edges[-1])
    names = sorted(set(d['specimen']))
    rate = sparse.csr_matrix(d['counts'].multiply((1e4 / d['library'])[:, None]))
    obs_rate, obs_sum, obs_n = [], [], []
    for n in names:
        for k in range(20):
            m = (d['specimen'] == n) & inside & (bins_ == k)
            obs_n.append(int(m.sum()))
            obs_rate.append(np.asarray(rate[m].mean(axis=0)).ravel() if m.any() else np.full(len(d['genes']), np.nan))
            obs_sum.append(np.asarray(d['counts'][m].sum(axis=0)).ravel())
    segs = sorted(set(d['segment']))
    seg_share = np.array([[((bins_ == k) & inside & (d['segment'] == s)).sum() / max(((bins_ == k) & inside).sum(), 1)
                           for k in range(20)] for s in segs])
    clusters = result.clusters.reset_index(drop=True)
    np.savez_compressed(
        tables / f's10_mechanics_{ds}.npz', genes=np.array(d['genes']), grid=grid, knots=model.knots, weights=np.asarray(result.weights),
        D=con.D.astype('float32'), P=np.asarray(con.P, 'float32'), N=np.asarray(con.N, 'float32'), S=np.asarray(sel.S, 'float32'),
        pair_names=np.array(list(map(str, con.pair_names))), null_names=np.array(list(map(str, con.null_names))),
        specimen_names=np.array(list(map(str, con.specimen_names))), specimen_filled=np.asarray(con.specimen_filled, 'float32'),
        fit_specimen_log2=np.asarray(fit.specimen, 'float32'),
        specimen_curves=np.array([model.specimen_curves[n] for n in names], 'float32'), curve_specimens=np.array(names),
        reference=model.reference.astype('float32'), case=model.case.astype('float32'), delta=model.delta.astype('float32'),
        level_reference=np.asarray(lvl['level_reference'], 'float32'), level_case=np.asarray(lvl['level_case'], 'float32'),
        bin_edges=edges, obs_specimens=np.repeat(names, 20), obs_bins=np.tile(np.arange(20), len(names)), obs_n=np.array(obs_n),
        obs_rate_per10k=np.array(obs_rate, 'float32'), obs_count_sum=np.array(obs_sum, 'float32'),
        segment_labels=np.array(segs), segment_share=seg_share,
        segment_n=np.array([int(((bins_ == k) & inside).sum()) for k in range(20)]),
        zones=ZONES[ds], size_factors=SF[ds].to_numpy(), size_factor_specimens=SF[ds].index.to_numpy().astype(str),
        cluster_table=clusters.astype(str).to_numpy().astype(str), cluster_columns=np.array(list(map(str, clusters.columns))))
    MECH_SUMMARY.append({'dataset': d['title'], 'selected': int(gt.selected.fillna(False).astype(bool).sum()),
                         'selected whole axis': int(gt.selected_whole_axis.fillna(False).astype(bool).sum()),
                         'shape replicated': int(gt.shape_replicated.fillna(False).astype(bool).sum()),
                         'pairs': len(con.P), 'mixed splits': len(con.N), 'M_level converged': float(np.mean(lvl['converged']))})
display(pd.DataFrame(MECH_SUMMARY).set_index('dataset'))
print('Primary and fast secondary sections done:', f'{time.perf_counter() - NOTEBOOK_START:.0f} s')

### S7 · Atlas positional replication by category (human vs mouse)

- **Atlas pseudobulks** per donor, early vs late PT: Census human (convoluted PT vs S3), Census mouse (S1 + S2 vs S3;
  embryonic and Theiler stages removed; ≥ 50 cells per segment; donors with every segment), Lake/KPMP (S1 + S2 vs S3,
  notebook 37's cached sums, checked against the h5ad obs; ≥ 50 nuclei per segment). Restricted to notebook 71's
  human vs mouse universe: a universe gene missing from an atlas table is absent there and drops out of that pair.
- **Ours:** per specimen, S1 + S2 vs S3 by the package labels, library = 71's mt-free library.
- **Gene test** for each pair (Census human vs male mouse, Lake human vs male mouse): genes present in both datasets,
  level ≥ 1 log2 CPM in each species, Census "other_id" symbols out; each donor's late library rescaled by
  2^median(log2((late+0.5)/lib) − log2((early+0.5)/lib)); limma voom paired (donor indicators + late + late × human),
  coefficient late × human; also limma-trend, Welch, Wilcoxon and the beyond-global versions.
- **Gene agreement (q2a):** among 71's q ≤ 0.05 genes, the share with population BH ≤ 0.05 and the same sign as 71's
  d_g; uncalled genes reweighted over 5 × 5 strata of our log mean count × the atlas mean level.
- **Pathways (q2):** cameraPR (0.01) on sets of ≥ 5 tested members (Ugt3a1/2 out): unsigned z = Φ⁻¹(1 − voom p), Up
  calls; signed voom t; aligned sign(d_g) · t; and the three beyond-global versions. **By category (q2b):** 71's
  categories (both, normalized only, clustering only, neither): tested, called, share, median z, and a one-sided
  Mann–Whitney test against "neither".

In [ ]:
import ast

import h5py
from scipy.stats import mannwhitneyu
from scipy.stats import t as tdist

MIN_CELLS, LEVEL_MIN = 50, 1.0
FLAG_GENES = ['Ugt3a1', 'Ugt3a2']
POP_GENES = pd.Index(DATA[hvm]['genes'])
POP_ROWS, POP_MATS = [], []


def pop_add(dataset, species, donor, sex, age, segment, counts, lib, n_cells):
    POP_ROWS.append(dict(dataset=dataset, species=species, donor=donor, sex=sex, age=age, segment=segment, lib=float(lib), n_cells=int(n_cells)))
    POP_MATS.append(pd.Series(counts, dtype=float).reindex(POP_GENES).to_numpy())


def census(label, segment_map, early_parts, dataset_name):
    table = pd.read_csv(census_dir / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(census_dir / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_CELLS)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    for (donor, sex, age), block in meta.groupby(['donor_id', 'sex', 'development_stage']):
        for seg, parts in (('late', ['S3']), ('early', early_parts)):
            idx = block.index[block.segment.isin(parts)]
            pop_add(dataset_name(sex), 'human' if label == 'human' else 'mouse', donor, sex, age, seg,
                    table.loc[idx].sum(), meta.loc[idx, 'total_counts_all_genes'].sum(), meta.loc[idx, 'n_cells'].sum())
    return [g for g in POP_GENES if g not in table.columns]


miss_h = census('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                          'epithelial cell of proximal tubule segment 3': 'S3'}, ['early'], lambda s: 'census_human')
miss_m = census('mouse', {'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
                          'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'], lambda s: f'census_mouse_{s}')
lake_files = sorted(lake_cache_dir.glob('lake_pseudobulk__*.npz'))
assert len(lake_files) == 1, lake_files
lake_key = json.loads(open(str(lake_files[0]) + '.key.json').read())
assert ast.literal_eval(lake_key['params'])['disease'] == 'normal'
lake_genes = ast.literal_eval(lake_key['inputs'])['genes']
with np.load(lake_files[0]) as z_:
    lake_sums = pd.DataFrame(z_['sums'], columns=lake_genes)
    lake_meta = pd.DataFrame([g.split('|') for g in z_['groups'].astype(str)], columns=['donor', 'sex', 'segment'])
    lake_meta['nuclei'], lake_meta['totals'] = z_['nuclei'], z_['totals']


def h5_column(group, name):
    item = group[name]
    if isinstance(item, h5py.Group):
        cats = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
        return pd.Categorical.from_codes(item['codes'][:], cats).astype(str)
    values = item[:]
    return values.astype(str) if values.dtype.kind in 'SO' else values


with h5py.File(lake_h5ad, 'r') as handle:
    cols = ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id', 'Age_binned', 'nCount_RNA']
    lake_obs = pd.DataFrame({c: h5_column(handle['obs'], c) for c in cols})
keep_lake = (lake_obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3']) & lake_obs.disease.eq('normal') & lake_obs.region.eq('C')
             & lake_obs.tissue.eq('cortex of kidney'))
check_lake = lake_obs[keep_lake].assign(segment=lake_obs.loc[keep_lake, 'subclass.l2'].str.replace('PT-', '', regex=False)) \
    .groupby(['donor_id', 'sex', 'segment']).agg(nuclei=('nCount_RNA', 'size'), ncount=('nCount_RNA', 'sum')).reset_index()
merged = lake_meta.merge(check_lake, left_on=['donor', 'sex', 'segment'], right_on=['donor_id', 'sex', 'segment'], how='outer')
assert (merged.nuclei_x == merged.nuclei_y).all(), 'the Lake cache disagrees with the h5ad'
lake_donor_meta = lake_obs[keep_lake].groupby('donor_id').agg(sex=('sex', 'first'), age=('Age_binned', 'first'))
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_CELLS).all(axis=1)]
for donor in lake_donors:
    for seg, parts in (('late', ['S3']), ('early', ['S1', 'S2'])):
        r_ = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.isin(parts)]
        pop_add('lake_human', 'human', donor, lake_donor_meta.loc[donor, 'sex'], lake_donor_meta.loc[donor, 'age'], seg,
                lake_sums.loc[r_].sum(), lake_meta.loc[r_, 'totals'].sum(), lake_meta.loc[r_, 'nuclei'].sum())
d = DATA[hvm]
for name in HUMAN + MOUSE:
    sp = 'human' if name in HUMAN else 'mouse'
    for seg, parts in (('late', ['S3']), ('early', ['S1', 'S2'])):
        m = (d['specimen'] == name) & np.isin(d['segment'], parts)
        pop_add(f'ours_{sp}', sp, name, 'male', 'ours', seg, pd.Series(np.asarray(d['counts'][m].sum(axis=0)).ravel(), index=POP_GENES),
                d['library'][m].sum(), m.sum())
POP_META = pd.DataFrame(POP_ROWS)
POP_X = pd.DataFrame(np.vstack(POP_MATS), columns=POP_GENES)
POP_ABSENT = {ds_: int(np.isnan(POP_X[POP_META.dataset.eq(ds_).to_numpy()].to_numpy()).all(axis=0).sum()) for ds_ in POP_META.dataset.unique()}
POP_DONORS = POP_META[POP_META.segment.eq('late')][['dataset', 'species', 'donor', 'sex', 'age']].merge(
    POP_META.pivot_table(index=['dataset', 'donor'], columns='segment', values='n_cells').reset_index(), on=['dataset', 'donor'])
POP_DONORS.to_csv(tables / 's7_donors.csv', index=False)
print('universe genes absent per atlas dataset:', POP_ABSENT)

audit = pd.read_csv(census_dir / 'census_symbol_audit.csv')
OTHER_ID = {sp: set(audit.loc[audit.species.eq(sp) & audit.status.eq('other_id'), 'column']) for sp in ('human', 'mouse')}

R_POP = '''
pop_voom <- function(counts, lib, design, coef) {
  v <- voom(counts, design, lib.size = lib)
  fit <- eBayes(lmFit(v, design))
  tt <- topTable(fit, coef = coef, number = Inf, sort.by = "none")
  list(t = tt$t, p = tt$P.Value, logFC = tt$logFC, ave = tt$AveExpr, d0 = fit$df.prior)
}
pop_trend <- function(counts, lib, design, coef) {
  lc <- t(log2(t(counts + 0.5) / (lib + 1) * 1e6))
  fit <- eBayes(lmFit(lc, design), trend = TRUE)
  tt <- topTable(fit, coef = coef, number = Inf, sort.by = "none")
  list(t = tt$t, p = tt$P.Value, logFC = tt$logFC, ave = tt$AveExpr, d0 = fit$df.prior)
}
'''
ro.r(R_POP)


def r_mat(x):
    x = np.asarray(x, float)
    return ro.r.matrix(ro.FloatVector(x.ravel(order='F')), nrow=x.shape[0], ncol=x.shape[1])


def limma_paired(counts, lib, donor, late, group, method='voom'):
    donor = np.asarray(donor).astype(str)
    X = np.column_stack([(donor == dn).astype(float) for dn in sorted(set(donor))] + [np.asarray(late, float), np.asarray(late, float) * np.asarray(group, float)])
    res = ro.r['pop_voom' if method == 'voom' else 'pop_trend'](r_mat(np.asarray(counts, float).T), ro.FloatVector(np.asarray(lib, float)),
                                                               r_mat(X), X.shape[1])
    return {k: np.asarray(res.rx2(k), float) for k in res.names}


def bh(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    out[ok] = false_discovery_control(p[ok]) if ok.any() else out[ok]
    return out


def welch(a, b):
    n1, n2 = a.shape[1], b.shape[1]
    v1, v2 = a.var(1, ddof=1) / n1, b.var(1, ddof=1) / n2
    t = (a.mean(1) - b.mean(1)) / np.sqrt(v1 + v2)
    df = (v1 + v2) ** 2 / (v1 ** 2 / (n1 - 1) + v2 ** 2 / (n2 - 1))
    return t, 2 * tdist.sf(np.abs(t), df)


def wilcoxon(a, b):
    both = np.concatenate([a, b], axis=1)
    ties = (np.diff(np.sort(both, axis=1), axis=1) == 0).any(axis=1)
    U, P = np.full(len(a), np.nan), np.full(len(a), np.nan)
    for mask_, method in ((~ties, 'exact'), (ties, 'asymptotic')):
        if mask_.any():
            res = mannwhitneyu(a[mask_], b[mask_], axis=1, alternative='two-sided', method=method)
            U[mask_], P[mask_] = res.statistic, res.pvalue
    return U, P


def pop_get(dataset):
    m = POP_META[POP_META.dataset.eq(dataset)]
    lm = m[m.segment.eq('late')]
    donors = list(lm.donor)
    em_idx = m.index[m.segment.eq('early')]
    em_idx = pd.Series(em_idx, index=m.loc[em_idx, 'donor']).loc[donors].to_numpy()
    return {'donors': donors, 'late': POP_X.loc[lm.index].to_numpy(float), 'early': POP_X.loc[em_idx].to_numpy(float),
            'lib_late': lm.lib.to_numpy(float), 'lib_early': POP_META.loc[em_idx, 'lib'].to_numpy(float)}


def l2(c, lib):
    return np.log2(c / lib[:, None] * 1e6 + 1)


def pop_level(dd):
    return np.maximum(l2(dd['late'], dd['lib_late']).mean(0), l2(dd['early'], dd['lib_early']).mean(0))


def pop_contrasts(dd, keep_):
    a = np.log2((dd['late'][:, keep_] + .5) / dd['lib_late'][:, None])
    b = np.log2((dd['early'][:, keep_] + .5) / dd['lib_early'][:, None])
    lib_late = dd['lib_late'] * 2 ** np.median(a - b, axis=1)
    return (l2(dd['late'][:, keep_], lib_late) - l2(dd['early'][:, keep_], dd['lib_early'])).T, lib_late


def run_pair(hds, mds, keep_):
    Dh, lh = pop_contrasts(hds, keep_)
    Dm, lm = pop_contrasts(mds, keep_)
    out = pd.DataFrame(index=POP_GENES[keep_])
    out['mean_h'], out['mean_m'] = Dh.mean(1), Dm.mean(1)
    out['effect'] = out.mean_h - out.mean_m
    out['welch_t'], out['welch_p'] = welch(Dh, Dm)
    out['wilcox_U'], out['wilcox_p'] = wilcoxon(Dh, Dm)
    nh, nm = len(hds['donors']), len(mds['donors'])
    counts_ = np.vstack([hds['late'][:, keep_], hds['early'][:, keep_], mds['late'][:, keep_], mds['early'][:, keep_]])
    lib_ = np.concatenate([lh, hds['lib_early'], lm, mds['lib_early']])
    donor = [f'h{i}' for i in range(nh)] * 2 + [f'm{i}' for i in range(nm)] * 2
    late = np.r_[np.ones(nh), np.zeros(nh), np.ones(nm), np.zeros(nm)]
    group = np.r_[np.ones(2 * nh), np.zeros(2 * nm)]
    for method in ('voom', 'trend'):
        r_ = limma_paired(counts_, lib_, donor, late, group, method=method)
        out[f'{method}_t'], out[f'{method}_p'], out[f'{method}_logFC'] = r_['t'], r_['p'], r_['logFC']
    gamma, alpha_ = np.polyfit(out.mean_m, out.mean_h, 1)
    out['resid_effect'] = out.mean_h - (alpha_ + gamma * out.mean_m)
    out['resid_welch_t'], out['resid_welch_p'] = welch(Dh, alpha_ + gamma * Dm)
    out['resid_wilcox_U'], out['resid_wilcox_p'] = wilcoxon(Dh, alpha_ + gamma * Dm)
    for c in [c for c in out.columns if c.endswith('_p')]:
        out[c.replace('_p', '_q')] = bh(out[c])
    return out


POP = {k: pop_get(k) for k in ('census_human', 'lake_human', 'census_mouse_male')}
present = {k: ~np.isnan(v['late']).any(0) & ~np.isnan(v['early']).any(0) for k, v in POP.items()}
for k in POP:
    POP[k]['late'], POP[k]['early'] = np.nan_to_num(POP[k]['late']), np.nan_to_num(POP[k]['early'])
PAIRS_POP = {'Census human vs male mouse': ('census_human', 'census_mouse_male'), 'Lake human vs male mouse': ('lake_human', 'census_mouse_male')}
GENE_TESTS = {}
for label, (h, m) in PAIRS_POP.items():
    keep_ = (present[h] & present[m] & (pop_level(POP[h]) >= LEVEL_MIN) & (pop_level(POP[m]) >= LEVEL_MIN)
             & ~POP_GENES.isin(list(OTHER_ID['human'] | OTHER_ID['mouse'])))
    res = run_pair(POP[h], POP[m], keep_)
    res['level_h'], res['level_m'] = pop_level(POP[h])[keep_], pop_level(POP[m])[keep_]
    GENE_TESTS[label] = res
    res.to_csv(tables / f"s7_gene_tests__{label.replace(' ', '_')}.csv")

# q2: our calls against the population tests
d = DATA[hvm]
ours = STAT[hvm]['species']
our_drift = D71[hvm]
sig_genes = ours.q.le(GENE_FDR)


def our_contrast(name):
    mm = POP_META[POP_META.donor.eq(name)].set_index('segment')
    rows_ = POP_X.loc[POP_META.index[POP_META.donor.eq(name)]].to_numpy()
    late, early = rows_[list(mm.index).index('late')], rows_[list(mm.index).index('early')]
    return pd.Series(np.log2(late / mm.loc['late', 'lib'] * 1e6 + 1) - np.log2(early / mm.loc['early', 'lib'] * 1e6 + 1), index=POP_GENES)


ours_d = pd.DataFrame({n: our_contrast(n) for n in HUMAN + MOUSE})
ours_int = ours_d[HUMAN].mean(1) - ours_d[MOUSE].mean(1)
ours_mouse = ours_d[MOUSE].mean(1)
our_logmean = np.log(POP_X.loc[POP_META.index[POP_META.dataset.str.startswith('ours')]].sum() / len(d['position']))
CATS = pd.Series({p: ('both' if p in N_CALLS[hvm]['species'] and p in b_calls(B[hvm][('species', None)]) else
                      'normalized only' if p in N_CALLS[hvm]['species'] else
                      'clustering only' if p in b_calls(B[hvm][('species', None)]) else 'neither') for p in d['sets']})
Q2A_ROWS, Q2B_ROWS, Q2 = [], [], {}
for pair in PAIRS_POP:
    pop = GENE_TESTS[pair]
    g_ = pop.index.intersection(ours.index)
    beta_ = np.polyfit(ours_mouse[g_], ours_int[g_], 1)
    ours_resid = ours_int - np.polyval(beta_, ours_mouse)
    tab = pd.DataFrame({'sig': sig_genes[g_], 'sign': np.sign(our_drift[g_]), 'sign_resid': np.sign(ours_resid[g_]),
                        'our_expr': our_logmean[g_], 'ext_expr': (pop.level_h + pop.level_m)[g_] / 2, 'flag': g_.isin(FLAG_GENES)})
    for test in ('voom', 'wilcox', 'trend', 'welch'):
        tab[f'{test}_hit'] = pop[f'{test}_q'][g_].le(.05) & (np.sign(pop.effect[g_]) == tab.sign)
        tab[f'{test}_any'] = pop[f'{test}_q'][g_].le(.05)
    tab['resid_hit'] = pop.resid_welch_q[g_].le(.05) & (np.sign(pop.resid_effect[g_]) == tab.sign_resid)
    tab['resid_any'] = pop.resid_welch_q[g_].le(.05)
    tab = tab[~tab.flag]
    tab['stratum'] = pd.qcut(tab.our_expr, 5, labels=False).astype(str) + '_' + pd.qcut(tab.ext_expr, 5, labels=False).astype(str)
    w_ = tab[tab.sig].stratum.value_counts(normalize=True)
    unc = tab[~tab.sig]
    for col, crit in (('voom_hit', 'voom BH<=0.05, same sign'), ('wilcox_hit', 'Wilcoxon BH<=0.05, same sign'),
                      ('trend_hit', 'limma-trend BH<=0.05, same sign'), ('welch_hit', 'Welch BH<=0.05, same sign'),
                      ('voom_any', 'voom BH<=0.05, either sign'), ('wilcox_any', 'Wilcoxon BH<=0.05, either sign'),
                      ('resid_hit', 'beyond global: Welch BH<=0.05, same sign as our residual'),
                      ('resid_any', 'beyond global: Welch BH<=0.05, either sign')):
        by = unc.groupby('stratum')[col].mean()
        Q2A_ROWS.append({'population pair': pair, 'criterion': crit, 'our significant genes testable': int(tab.sig.sum()),
                         'share of our significant genes': float(tab.loc[tab.sig, col].mean()),
                         'share of uncalled genes (expression-matched)': float((w_ * by.reindex(w_.index)).sum()),
                         'share of uncalled genes (raw)': float(unc[col].mean()), 'uncalled genes testable': int(len(unc)),
                         'population calls that are our significant genes': float(tab.loc[tab[col], 'sig'].mean())})
    stats_ = {'unsigned': pd.Series(norm.isf(np.clip(pop.voom_p.to_numpy(float), 1e-300, 1 - 1e-16)), index=pop.index),
              'signed': pop.voom_t, 'aligned': np.sign(our_drift.reindex(pop.index)) * pop.voom_t,
              'beyond global, unsigned': pd.Series(norm.isf(np.clip(pop.resid_welch_p.to_numpy(float), 1e-300, 1 - 1e-16)), index=pop.index),
              'beyond global, signed': pop.resid_welch_t,
              'beyond global, aligned': np.sign(ours_resid.reindex(pop.index)) * pop.resid_welch_t}
    restricted = {p: [x for x in v if x in set(pop.index) and x not in FLAG_GENES] for p, v in d['sets'].items()}
    restricted = {p: v for p, v in restricted.items() if len(v) >= 5}
    out = pd.DataFrame(index=list(restricted))
    out['cat'] = CATS.reindex(out.index)
    out['n_members_tested'] = [len(restricted[p]) for p in out.index]
    for name, s in stats_.items():
        s = s.dropna()
        s = s[~s.index.isin(FLAG_GENES)]
        cam = camera_pr(s, restricted, inter_gene_cor=CAMERA_COR).set_index('pathway_id')
        out[f'{name}: z'] = pd.Series(np.where(cam.direction.eq('Up'), 1, -1) * norm.isf(np.clip(cam.p, 1e-300, 1) / 2), index=cam.index)
        out[f'{name}: fdr'] = cam.fdr
        out[f'{name}: direction'] = cam.direction
        signed = name.endswith('signed') and not name.endswith('unsigned')
        out[f'{name}: called'] = cam.fdr.le(.05) if signed else cam.fdr.le(.05) & cam.direction.eq('Up')
    Q2[pair] = out
    out.rename_axis('pathway_id').to_csv(tables / f"s7_q2_pathways__{pair.replace(' ', '_')}.csv")
    ref_ = out[out.cat.eq('neither')]
    for cat, block in out.groupby('cat'):
        row = {'population pair': pair, 'category': cat, 'pathways tested': len(block)}
        for name in stats_:
            row[f'{name}: called'] = int(block[f'{name}: called'].sum())
            row[f'{name}: share called'] = float(block[f'{name}: called'].mean())
            row[f'{name}: median z'] = float(block[f'{name}: z'].median())
            if cat != 'neither':
                row[f'{name}: MW p vs neither (z greater)'] = mannwhitneyu(block[f'{name}: z'], ref_[f'{name}: z'], alternative='greater').pvalue
        Q2B_ROWS.append(row)
Q2A = pd.DataFrame(Q2A_ROWS)
Q2B = pd.DataFrame(Q2B_ROWS)
Q2A.to_csv(tables / 's7_q2a_gene_level.csv', index=False)
Q2B.to_csv(tables / 's7_q2b_pathway_by_category.csv', index=False)
display(Q2A[Q2A.criterion.eq('voom BH<=0.05, same sign')].set_index('population pair').round(3))
display(Q2B.set_index(['population pair', 'category'])[['pathways tested', 'unsigned: called', 'unsigned: share called',
                                                          'unsigned: median z', 'unsigned: MW p vs neither (z greater)']].round(4))

### S8 · Atlas level replication (human vs mouse)

- **Per-donor level:** 2/3 · log2CPM(early) + 1/3 · log2CPM(late), from the S7 pseudobulks.
- **Donors:** Lake (7) or Census human (6) against the 6 Census males aged 3–6 or 12 months.
- **Genes:** mean ≥ 1 in either species, no missing value, Ugt3a1/2 out. **Fit:** limma `~species`, eBayes trend and
  robust. **Set tests** (sets ≥ 5 members): camera with inter.gene.cor NA, camera with 0.01 (both trend.var), and ORA
  by direction on BH ≤ 0.05 genes (background the kept genes, BH over set × direction, best row per set).
- **Replication table:** sets called in both atlases, as "called in both / tested" per group (71's clustering calls,
  clustering only, normalized calls, S3's level ORA calls, neither).
- **Null:** 40 random 6 v 6 splits of the 12 Census males (`default_rng(11)`), camera with estimated correlation.

In [ ]:
def level_matrix(dataset):
    m = POP_META[POP_META.dataset.eq(dataset)]
    out = {}
    for donor, blk in m.groupby('donor'):
        v = {s: np.log2(POP_X.loc[blk.index[blk.segment.eq(s)][0]].to_numpy(float) / blk.loc[blk.index[blk.segment.eq(s)][0], 'lib'] * 1e6 + 1)
             for s in ('early', 'late')}
        out[donor] = 2 / 3 * v['early'] + 1 / 3 * v['late']
    return pd.DataFrame(out, index=POP_GENES)


def r_rows(df):
    return ro.r.matrix(ro.FloatVector(df.to_numpy().ravel()), nrow=df.shape[0], byrow=True)


def atlas_level(human, mouse, null=False):
    E = pd.concat([human, mouse], axis=1)
    keep_ = ((human.mean(axis=1) >= 1) | (mouse.mean(axis=1) >= 1)) & E.notna().all(axis=1) & ~E.index.isin(FLAG_GENES)
    E = E[keep_]
    genes_ = E.index
    grp = np.r_[np.ones(human.shape[1]), np.zeros(mouse.shape[1])]
    design = ro.r.matrix(ro.FloatVector(np.r_[np.ones(len(grp)), grp]), nrow=len(grp))
    rE = r_rows(E)
    fit_ = limma.eBayes(limma.lmFit(rE, design), trend=True, robust=True)
    tt = limma.topTable(fit_, coef=2, number=len(genes_), sort_by='none')
    t_, q_ = np.asarray(tt.rx2('t')), np.asarray(tt.rx2('adj.P.Val'))
    idx = {k: [genes_.get_loc(g) + 1 for g in v if g in genes_] for k, v in DATA[hvm]['sets'].items()}
    idx = {k: v for k, v in idx.items() if len(v) >= 5}
    rsets = ro.ListVector({k: ro.IntVector(v) for k, v in idx.items()})
    res = {}
    for name, cor in (('camera, correlation estimated', ro.NA_Real), ('camera, correlation 0.01', .01)):
        c = limma.camera(rE, rsets, design, contrast=2, trend_var=True, sort=False, **{'inter.gene.cor': cor})
        dr = c.rx2('Direction')
        dirn = [dr.levels[i - 1] for i in dr] if hasattr(dr, 'levels') else list(dr)
        res[name] = pd.DataFrame({'fdr': np.asarray(c.rx2('FDR')), 'direction': dirn}, index=list(idx))
    if not null:
        N_, recs = len(genes_), []
        for direction, sel_ in (('Up', (q_ <= .05) & (t_ > 0)), ('Down', (q_ <= .05) & (t_ < 0))):
            K = int(sel_.sum())
            for k, v in idx.items():
                x = int(sel_[np.array(v) - 1].sum())
                recs.append((k, direction, hypergeom.sf(x - 1, N_, K, len(v))))
        o = pd.DataFrame(recs, columns=['pathway_id', 'direction', 'p'])
        o['fdr'] = multipletests(o.p, method='fdr_bh')[1]
        res['ORA by direction (q<=0.05 genes)'] = o.sort_values('p').drop_duplicates('pathway_id').set_index('pathway_id')[['fdr', 'direction']]
        res['genes'] = len(genes_)
    return res


mice = level_matrix('census_mouse_male')
adult = POP_DONORS[(POP_DONORS.dataset == 'census_mouse_male') & POP_DONORS.age.str.contains('3-6|12-month')].donor.tolist()
assert len(adult) == 6, adult
ATLAS_LEVEL = {k: atlas_level(level_matrix(v), mice[adult]) for k, v in (('Lake', 'lake_human'), ('Census human', 'census_human'))}
for atlas, res in ATLAS_LEVEL.items():
    pd.concat({k: v for k, v in res.items() if k != 'genes'}, axis=1).rename_axis('pathway_id').to_csv(
        tables / f"s8_atlas_level_{atlas.replace(' ', '_')}.csv")
clu71 = b_calls(B[hvm][('species', None)])
GROUPS8 = {f'clustering calls ({len(clu71)})': clu71, f"clustering only ({int(CATS.eq('clustering only').sum())})": set(CATS.index[CATS.eq('clustering only')]),
           f"normalized shape calls ({len(N_CALLS[hvm]['species'])})": N_CALLS[hvm]['species'],
           f'our level list+ORA calls ({len(S3_CALLS)})': S3_CALLS, f"neither ({int(CATS.eq('neither').sum())})": set(CATS.index[CATS.eq('neither')])}
S8_ROWS = []
for test in ('camera, correlation estimated', 'camera, correlation 0.01', 'ORA by direction (q<=0.05 genes)'):
    both_ = None
    for atlas in ATLAS_LEVEL:
        called = set(ATLAS_LEVEL[atlas][test].index[ATLAS_LEVEL[atlas][test].fdr.le(.05)])
        both_ = called if both_ is None else both_ & called
    row = {'atlas test': test, 'Lake calls': int(ATLAS_LEVEL['Lake'][test].fdr.le(.05).sum()),
           'Census human calls': int(ATLAS_LEVEL['Census human'][test].fdr.le(.05).sum()), 'called in both atlases': len(both_)}
    for gname, ids_ in GROUPS8.items():
        ids_ = [i for i in ids_ if i in ATLAS_LEVEL['Lake'][test].index]
        row[gname] = f'{sum(i in both_ for i in ids_)}/{len(ids_)}'
    S8_ROWS.append(row)
S8 = pd.DataFrame(S8_ROWS).set_index('atlas test').T
S8.to_csv(tables / 's8_atlas_level_replication.csv')
display(S8)


def null_splits():
    rng = np.random.default_rng(11)
    allm, calls_ = mice.columns.tolist(), []
    for _ in range(40):
        a_ = list(rng.choice(allm, 6, replace=False))
        b_ = [x for x in allm if x not in a_]
        calls_.append(int(atlas_level(mice[a_], mice[b_], null=True)['camera, correlation estimated'].fdr.le(.05).sum()))
    return {'calls': np.array(calls_)}


S8_NULL = cached('s8_null_splits', null_splits, params={'splits': 40, 'seed': 11},
                 inputs={'mice': mice.to_numpy(), 'genes': list(POP_GENES), 'sets': DATA[hvm]['sets']})
pd.DataFrame({'split': np.arange(40), 'camera calls (correlation estimated)': S8_NULL['calls']}).to_csv(tables / 's8_null_splits.csv', index=False)
print(f"S8 null: 6 v 6 male-mouse splits with any camera call: {int((S8_NULL['calls'] > 0).sum())} of 40")

### S9 · Winsorized sensitivity

- **Cap:** from notebook 71's condition-split M_full and α, y' = min(y, μ + c·√(μ + αμ²)) on the nonzero counts, with
  c ∈ {8, 8f, 4, 4f}; in "f" variants the cap is never below 1 count. Capped counts are not rounded; the library is
  unchanged.
- **Rerun:** the full positional F-test (gene model with α re-estimated, the relabelings with that α, the
  denominator, `moderated_f` with covariate log of the winsorized mean, BH, cameraPR). The clustering method is not refit.
- **Summary:** against notebook 71's own calls: condition calls, kept/lost/new, relabeled calls, calibration share,
  gene-level agreement.

In [ ]:
WINSOR_C = ('8', '8f', '4', '4f')


def winsorize(counts, x, beta, alpha_, offset, c, floor):
    y = sparse.csc_matrix(counts, dtype=float, copy=True)
    capped_n, removed = np.zeros(y.shape[1], int), np.zeros(y.shape[1])
    for g_ in range(y.shape[1]):
        s_, e_ = y.indptr[g_], y.indptr[g_ + 1]
        rows_ = y.indices[s_:e_]
        mu = np.clip(np.exp(x[rows_] @ beta[g_] + offset[rows_]), 1e-12, 1e8)
        cap = np.maximum(mu + c * np.sqrt(mu + alpha_[g_] * mu ** 2), floor)
        v = y.data[s_:e_]
        over = v > cap
        capped_n[g_], removed[g_] = int(over.sum()), float((v[over] - cap[over]).sum())
        v[over] = cap[over]
    return y.tocsr(), capped_n, removed


def winsor_run(ds, arg):
    d, model = DATA[ds], MODELS[ds]['species']
    c, floor = float(arg.rstrip('f')), (1. if arg.endswith('f') else 0.)
    offset = np.log(d['library'])
    group = PARTITIONS[ds]['species'][0]
    designs, w = contrast_designs(d['position'], group, d['specimen'], model.knots)
    fit_ = nb_fit(d['counts'], designs['full'], w, offset, model.dispersion, n_jobs=N_JOBS)
    yw, capped_n, removed = winsorize(d['counts'], designs['full'], fit_['beta'], np.asarray(model.dispersion['alpha'], float), offset, c, floor)
    log_mean = np.log(np.asarray(yw.mean(axis=0)).ravel())
    out, models_ = {'capped_n': capped_n, 'removed': removed, 'nonzero': np.array(d['counts'].nnz)}, {}
    for k, (label, (grp, nuis)) in enumerate(PARTITIONS[ds].items()):
        disp = None if label == 'species' else models_['species'].dispersion
        models_[label] = m_ = standard_gene_model(yw, d['library'], d['position'], grp, d['specimen'], genes=d['genes'], nuisance_shape=nuis,
                                                  dispersion=disp, basis_df=BASIS_DF, n_jobs=N_JOBS)
        den = specimen_shape_lr(yw, d['library'], d['position'], grp, d['specimen'], models_['species'].dispersion, nuisance_shape=nuis,
                                basis_df=BASIS_DF, n_jobs=N_JOBS)
        assert den['df2'] == (12 if label == 'species' else 6)
        fr = moderated_f(m_.statistics.T_spatial, DF1, np.asarray(den['LR_spec'], float), den['df2'], covariate=log_mean)
        t_ = camera_pr(fr.z, d['sets'], inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
        out.update({f'{k}|z': fr.z.to_numpy(), f'{k}|p': fr.p.to_numpy(), f'{k}|q': false_discovery_control(fr.p.to_numpy()),
                    f'{k}|converged': (m_.statistics.converged.to_numpy() & np.asarray(den['converged'], bool)),
                    f'{k}|set_p': t_.p.to_numpy(), f'{k}|set_p_one_sided': t_.p_one_sided.to_numpy(), f'{k}|set_fdr': t_.fdr.to_numpy(),
                    f'{k}|set_call': t_.call.to_numpy(bool)})
    out['alpha'] = np.asarray(models_['species'].dispersion['alpha'], float)
    return out


S9_ROWS, S9_DETAIL = [], []
for ds, d in DATA.items():
    labels = list(PARTITIONS[ds])
    ids = list(d['sets'])
    calls0 = N_CALLS[ds]['species']
    share0 = float(np.mean(np.concatenate([TESTS_N[ds][l].p_one_sided.to_numpy() for l in labels[1:]]) <= ALPHA))
    S9_ROWS.append({'dataset': d['title'], 'c': 'none (notebook 71)', 'condition_calls': len(calls0),
                    'relabeled_calls': f"{len(N_CALLS[ds][labels[1]])}, {len(N_CALLS[ds][labels[2]])}", 'calibration_share': share0,
                    'gene_BH05_condition': int((STAT[ds]['species'].q <= .05).sum())})
    for arg in WINSOR_C:
        wr = cached(f's9_winsor_{ds}_c{arg}', lambda ds=ds, arg=arg: winsor_run(ds, arg), params={'c': arg, 'basis_df': BASIS_DF},
                    inputs={**d['input_key'], 'alpha': MODELS[ds]['species'].dispersion['alpha'], 'sets': d['sets']})
        calls1 = set(np.array(ids)[wr['0|set_call'].astype(bool)])
        share1 = float(np.mean(np.concatenate([wr[f'{k}|set_p_one_sided'] for k in (1, 2)]) <= ALPHA))
        ok = STAT[ds]['species'].converged.to_numpy() & wr['0|converged'].astype(bool)
        q0, q1 = STAT[ds]['species'].q.to_numpy() <= .05, wr['0|q'] <= .05
        S9_ROWS.append({'dataset': d['title'], 'c': arg, 'entries_capped': int(wr['capped_n'].sum()),
                        'share_nonzero_capped': float(wr['capped_n'].sum() / int(wr['nonzero'])),
                        'count_mass_removed_share': float(wr['removed'].sum() / d['counts'].sum()),
                        'median_alpha_before_after': f"{np.median(MODELS[ds]['species'].dispersion['alpha']):.4f} -> {np.median(wr['alpha']):.4f}",
                        'gene_z_spearman_condition': float(spearmanr(STAT[ds]['species'].z.to_numpy()[ok], wr['0|z'][ok])[0]),
                        'gene_BH05_condition': int(q1.sum()), 'gene_BH05_kept': int((q0 & q1).sum()),
                        'condition_calls': len(calls1), 'kept': len(calls0 & calls1), 'lost': len(calls0 - calls1), 'new': len(calls1 - calls0),
                        'relabeled_calls': f"{int(wr['1|set_call'].sum())}, {int(wr['2|set_call'].sum())}", 'calibration_share': share1,
                        'pathway_p_spearman': float(spearmanr(TESTS_N[ds]['species'].p, wr['0|set_p'])[0]),
                        'seconds': round(SECONDS[f's9_winsor_{ds}_c{arg}'], 1)})
        fdr0 = TESTS_N[ds]['species'].set_index('pathway_id').fdr
        fdr1 = pd.Series(wr['0|set_fdr'], index=ids)
        for status, members in (('lost', calls0 - calls1), ('new', calls1 - calls0)):
            for p in sorted(members):
                S9_DETAIL.append({'dataset': d['title'], 'c': arg, 'status': status, 'pathway_id': p, 'fdr_before': fdr0[p], 'fdr_after': fdr1[p]})
S9 = pd.DataFrame(S9_ROWS)
S9.to_csv(tables / 's9_winsor_summary.csv', index=False)
pd.DataFrame(S9_DETAIL).to_csv(tables / 's9_winsor_lost_new.csv', index=False)
display(S9.set_index(['dataset', 'c']).round(4))

## 13 · Readings of the protocol

Where the protocol left a choice, the reading closest to notebook 69 was taken, before any fit.
- **Positions.** The stage-04 value, trimmed to every specimen's [1st, 99th] percentile common support and min–max
  scaled over the kept units (protocol section 3). Notebook 69's own sets and positions (unit matching) use 69's
  rules: SCF13 as `pt_inputs` scaled it, the AKI DPT min–max over 69's support.
- **The AKI pool** is `hvm_inputs`' universe with the mt genes kept, as notebook 69 passed notebook 12's universe.
- **The "69 ·" rows** of the method table are notebook 69's own rows (its calls, not 66/67's).
- **Check (d)'s notebook 69 rows** (including its proteasome sensitivity) are copied from notebook 69's table.
- **The null genes of the figure and of the 69/71 drift column** are genes tested in both runs, converged and not
  separated in both, with q > 0.5 in both; each run is drawn on its own position scale.
- **M6.** 69's called redundancy clusters are 69's clusters (its `redundancy_clusters` table) touched by 69's
  normalized calls among sets tested in both runs; one is "called again" when a 71 call among those sets falls in it.
  A five-strongest call that is not tested in 71 counts as lost.
- **Pathway comparison.** Notebook 69's sets are rebuilt by its own calls (`pt_inputs` for human vs mouse, its AKI
  helper on notebook 12's universe; guarded against its tables);
  median d_g uses each run's own members and flags.
- **Gene tables** add T_spatial and LR_spec to notebook 69's columns (notebook 72 reads T_spatial).
- **S1.** α is the condition split's; the shape arm is the primary z (guarded). GSEA uses `threads` = the worker
  count; FDR and direction are saved for every arm. Overlap (S6) also lists the GSEA arms.
- **S3.** `logmean` is the log of 71's mean raw count per unit (the script's construction on 71's data).
- **S7.** The Lake pseudobulk is notebook 37's cache, so universe genes outside notebook 37's universe are absent
  from the Lake pair. Only the two male-mouse pairs are run (the page uses only those).
- **S8.** The null splits are saved as a table.
- **S9.** The cap uses 71's own condition-split M_full and α; the summary compares with 71's own calls.
- **S10.** Curves are saved on all 101 grid points (a superset of "every second point").
- **Not run:** notebook 66's raw-`T_spatial` GSEA, the joint test and the set-curve calls; notebook 68 and the nb34
  external table (older statistic and coordinate).

## 14 · Summary

**Inputs.** Human vs mouse: 11,947 package PT units, 11,383 in the common support, 11,055 genes, 1,513 pathways.
AKI: 15,013 units, 14,048 in the support, 10,198 genes, 1,469 pathways (40 of them tested only here). Every guard
passed, including notebook 70's confirmed pass-1 fingerprints. The AKI universe still lacks Havcr1, Lcn2, Mki67 and
Top2a: they are absent from the human vs mouse universe, although the AKI detection rule alone would admit them
(equal-specimen detection 46%, 25%, 9% and 9%).

**Material-change verdicts (protocol section 7): none of M1–M7 is met.**

| Rule | Notebook 69 | Notebook 71 |
|---|---|---|
| M1 F-test, human vs mouse | 97 calls; 0, 0; calibration 0.044; passes | 83 calls (77 clusters); 1, 0; NCDR 0.006; calibration 0.071; passes |
| M2 F-test, AKI (relabeling, calibration) | pass, fail (NCDR 0.13; 0.101) | pass, fail (103 calls; 13, 29; NCDR 0.20; 0.120) |
| M3 clustering fair null | 0, 0 and 2, 0 | 0, 0 and 0, 0 |
| M4 L1 (L2) | 0.704 (0.083); AKI 0.549 (0.431) | 0.708 (0.138); AKI 0.563 (0.349) |
| M5 Jaccard over redundancy clusters | 0.175; AKI 0.096 | 0.171; AKI 0.121 |
| M6 69's called clusters called again; five strongest | — | 76 of 90 (0.84); all five called |
| M7 median d_g, own q > 0.5 genes | +0.001 | +0.006 (AKI +0.009) |

- **Primary calls.** The positional F-test calls 83 human vs mouse pathways and 103 AKI pathways. The human vs mouse
  calibration share (0.071) sits close to its limit of 0.075. The AKI calibration failure of notebooks 67–69 stands.
- **The clustering comparator** calls 94 human vs mouse pathways (69: 132) and 459 AKI pathways (69: 455), still
  specific under the fair null. Its single-split nulls give 97 and 152 human vs mouse calls (69: 162 and 46).

**Against notebook 69** (descriptive):
- **Units.** 93–100% of the package units match an old structure. The position Spearman is 0.91–0.96 in the mice and
  0.87 in the human sections. Segment labels agree for 80–82% of human units and 86–96% of mouse units: 947 of the
  3,404 old human vs mouse S2 units are now S1.
- **Genes.** The Spearman of the condition z is 0.825 (human vs mouse) and 0.880 (AKI); of d_g 0.913 and 0.945. The
  Jaccard of the q ≤ 0.05 genes is 0.70 and 0.77.
- **Normalized pathways.** Human vs mouse keeps 82, loses 15 and gains 1 (Jaccard 0.84; Spearman of the cameraPR p
  0.83). All 15 lost calls had a notebook 69 FDR between 0.013 and 0.048. AKI keeps 84, loses 15 and gains 18
  (Jaccard 0.72); the gains include oxidative phosphorylation and respiratory electron transport.
- **Clustering pathways** move more. Human vs mouse keeps 67, loses 65 and gains 27 (Jaccard 0.42); AKI keeps 363, loses 92 and gains 96 (Jaccard 0.66).
- **Why the human vs mouse clustering calls moved.** The list "Down · gradient toward S3" carried 107 of
  notebook 69's calls and carries none now. The proteasome subunits sit in D0 (4), D4 (3) and D5 (1), and the
  list carrying most calls is cluster D4 (198 genes, 58 calls).
- **Without the 8 proteasome subunits** the human vs mouse clustering calls fall from 94 to 43 (69: 132 to 34). The
  normalized calls go from 83 to 84.

**Secondary sections** (pre-specified; descriptive):
- **S1.** Only the primary arm passes. On human vs mouse, shape GSEA (75 calls, calibration 0.144), the level arms
  (cameraPR 20 calls with relabeled 102 and 149; GSEA 115) and the total arms (38 and 38) all fail the swap rule.
  Every AKI arm fails.
- **S2.** ssGSEA + limma passes on both datasets:
  - human vs mouse: level 1,208 calls (21, 0; 0.043), shape 866 (0, 0; 0.040);
  - AKI: level 1,178 (14, 0; 0.074), shape 1,117 (0, 0; 0.029).
- **S3.** The level gene list + ORA calls 114 pathways, with 0 and 0 under the swaps (10,405 genes kept). It shares 61
  calls with clustering (Jaccard 0.42; 69: 92 and 0.59).
- **S5.** The implied inter-gene correlation under the swaps is 0.003–0.006 (shape) and 0.056–0.069 (level) on human
  vs mouse; 0.024–0.029 and 0.033–0.096 on AKI.
- **S6.** AKI level cameraPR shares 358 calls with clustering (Jaccard 0.55); the direction agrees in 357 of 358.
- **S7 (atlases).** Of our q ≤ 0.05 genes, 53% (Census) and 56% (Lake) are population-significant with the same
  sign, against 20% and 22% of matched uncalled genes. Pathways called by the atlas positional test:
  - both methods: 4 of 19;
  - normalized only: 10 (Census) and 12 (Lake) of 64;
  - clustering only: 0 of 75;
  - neither: 7–8 of 1,351.
- **S8 (atlas level).** 11 sets are called by both atlases' level ORA. Among them: 6 of 94 clustering calls, 3 of 83
  normalized calls and 7 of 114 level-ORA calls. Camera with estimated correlation calls nothing, nor in any of 40
  null splits.
- **S9 (winsorizing).** Human vs mouse keeps 79–83 of the 83 calls (83–89 calls; swaps ≤ 4, 2; calibration 0.061–0.073).
  AKI keeps 94–102 of 103, and its calibration stays 0.116–0.118.

**What this cannot tell you.**
- With two specimens per group every result is descriptive. The two human "replicates" are sections of one donor.
- The two runs differ in units, counts, labels and coordinate at once: no difference here is attributable to the
  coordinate alone.
- The human vs mouse calibration share (0.071) is close to its limit.
- Under the package's labels the most injured AKI units are not in the PT cohort. The AKI universe excludes the
  injury markers through the human vs mouse universe rule.

In [ ]:
print('Material-change verdicts:')
for _, r_ in MATERIAL.iterrows():
    print(f"  {r_['rule']}: {'MATERIAL CHANGE' if r_['material change'] else 'no material change'} · {r_['what']}: {r_['notebook 71']}")
for ds in DATASETS:
    rows = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')
    for label in (N_LABEL, B_LABEL):
        r_ = rows.loc[label]
        print(f"{DATA[ds]['title']} · {label}: {r_['calls']} calls ({r_['calls (redundancy clusters)']} clusters); relabeled "
              f"{r_['relabeled calls']}; NCDR {r_['NCDR']:.2f} ({r_['relabeling rule']}); calibration {r_['calibration share']:.3f} "
              f"({r_['calibration']})")

## 15 · Run record

In [ ]:
import platform

from rpy2.robjects import r as R

cache_entries = []
for sidecar in sorted(cache.glob('*.key.json'), key=lambda p: json.loads(p.read_text())['written_utc']):
    meta = json.loads(sidecar.read_text())
    if meta['stage'] in SECONDS:
        cache_entries.append({'stage': meta['stage'], 'key': meta['key'], 'written_utc': meta['written_utc'],
                              'compute seconds': round(SECONDS[meta['stage']], 1)})
CACHE_TABLE = pd.DataFrame(cache_entries).drop_duplicates('stage', keep='last').sort_values('stage').reset_index(drop=True)
display(CACHE_TABLE)
record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
    'protocol': 'docs/results/pt-package-pseudospace.md, Notebook 71 protocol',
    'inputs': {ds: PROVENANCE[ds] for ds in DATASETS}, 'datasets': INPUT_TABLE.to_dict(), 'n_jobs': N_JOBS, 'mt_genes': MT,
    'excluded': {ds: {'library': DATA[ds]['mt_library'], 'universe': DATA[ds]['mt_universe']} for ds in DATASETS},
    'gene_model': {'module': 'pseudospace.pathway_pipelines.standard_gene_model', 'basis_df': BASIS_DF,
                   'grid_points': GRID_POINTS, 'nb_gam': nb_gam.GENE_MODEL},
    'normalized': {'denominator': 'pathway_pipelines.specimen_shape_lr', 'df2': DF2, 'squeezeVar covariate': 'log mean count',
                   'cameraPR': {'inter.gene.cor': CAMERA_COR, 'use.ranks': False}},
    'clustering': {'package': EXTERNAL, 'compare_params': repr(COMPARE_PARAMS), 'pathway_params': repr(PATHWAY_PARAMS),
                   'node_spacing': NODE_SPACING, 'floor_counts': FLOOR_COUNTS, 'zones': {ds: ZONES[ds].tolist() for ds in DATASETS},
                   'runs': RUN_TABLE.to_dict(orient='records')},
    'method_table': METHOD_TABLE.to_dict(orient='records'), 'level_dominance': DOMINANCE.to_dict(orient='records'),
    'material_change': MATERIAL.to_dict(orient='records'), 'primary_passes': PASS, 'guards': guard_log,
    'secondary': {'s1': S1_PASS.to_dict(orient='records'), 's2': S2_PASS.to_dict(orient='records'),
                  's3': S3_SUMMARY.reset_index().to_dict(orient='records'), 's3_genes': S3_DROPPED,
                  's7_absent_genes': POP_ABSENT, 's8_null_calls': S8_NULL['calls'].tolist(), 's9': S9.to_dict(orient='records')},
    'versions': {'python': platform.python_version(), 'R': str(R('R.version.string')[0]),
                 'limma': str(R('as.character(packageVersion("limma"))')[0]),
                 **{m: version(m) for m in ('numpy', 'scipy', 'pandas', 'rpy2', 'joblib', 'matplotlib', 'gseapy', 'pseudospace-reconstruction')}},
    'stage_cache': CACHE_TABLE.to_dict(orient='records'), 'compute_seconds_total': float(sum(SECONDS.values())),
    'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s; summed uncached compute time of the cached stages "
      f"{record['compute_seconds_total']:.0f} s")